# Hawkes $k$-Point Function — Full Pipeline Demo

**Model:** Nonlinear Hawkes process, 2 populations, delta-function synaptic filter

Set `k` and `max_ell` in the configuration cell below to control which
cumulant and loop orders are computed.  All intermediate results are
cached so subsequent runs with the same parameters are instantaneous.

In [ ]:
%display latex
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from IPython.display import display, Math
from sage.all import (
    SR, matrix, latex, I, pi, exp, diff, solve, integrate, oo,
    dirac_delta, function, var
)

from msrjd.core.field_theory import FieldTheory, fourier_transform, inverse_fourier_transform
from msrjd.core.vertices import extract_vertex_types, extract_source_types, available_degrees
from msrjd.core.cache import PipelineCache
from msrjd.enumeration.loop_diagram_enumeration import enumerate_all
from msrjd.diagrams.filter import filter_prediagrams, classify_prediagram_vertices
from msrjd.diagrams.type_assignment import (
    enumerate_typed_diagrams, enumerate_all as enumerate_all_typed,
    build_field_index_map, TypedDiagram
)
from msrjd.diagrams.causality import filter_causal
from msrjd.diagrams.symmetry import (
    combinatorial_factor, compute_all_combinatorial_factors,
    deduplicate_typed_diagrams,
)

from models.hawkes_linear_sage import HAWKES_MODEL

print('Imports loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════
k       = 3      # k-point cumulant (number of external legs)
max_ell = 1      # maximum loop order (0 = tree only, 1 = tree + 1-loop, ...)

# External fields: one per external leg (must have length k).
# Pick from the physical fields of the model: ('dn', pop), ('dv', pop)
# where pop is a population index (1, 2, ...).
# Examples:
#   k=1:  [('dn', 1)]
#   k=2:  [('dn', 1), ('dn', 2)]
#   k=2:  [('dn', 1), ('dv', 1)]    (mixed correlator)
external_fields = [('dn', 1), ('dn', 1), ('dn', 2)]

USE_CACHE = True  # True: pull from cache when available; False: recompute all

assert len(external_fields) == k, f'external_fields has {len(external_fields)} entries but k={k}'
print(f'k = {k},  max loop order = {max_ell}')
print(f'External fields: {external_fields}')

---
## 1. Theory / Model Information

In [ ]:
m = HAWKES_MODEL
print(f"Model:  {m['name']}")
print(f"Convention: {m['background_rate_convention']}")
print()

print('── Index sets ──')
for name, vals in m['index_sets'].items():
    print(f'  {name}: {list(vals)}')
print()

print('── Response fields (not expanded — full integration variables) ──')
for f in m['response_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Physical fields (expanded around MF background) ──')
for f in m['physical_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Parameters ──')
for p in m.get('parameters', []):
    idx = '(indexed)' if p.get('indexed') else '(scalar)'
    dom = f", domain={p['domain']}" if p.get('domain') else ''
    print(f"  {p['name']}  {idx}{dom}  — {p.get('description', '')}")
print()

print('── Kernels ──')
for kern in m.get('kernels', []):
    print(f"  {kern['name']}  — {kern.get('description', '')}")
print()

print('── Operators ──')
for o in m.get('operators', []):
    print(f"  {o['name']}  (latex: ${o.get('latex_name', o['name'])}$)  — {o.get('description', '')}")
print()

print('── Nonlinear functions ──')
for f in m.get('functions', []):
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Specializations ──')
print('  φ quadratic (cubic and higher coefficients = 0)')
print('  g = δ(t)  (instantaneous synaptic coupling)')

### 1.1 Expand the action and show all sectors

In [3]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print(f'Taylor order: {ft.taylor_order}')
print(f'Polynomial ring: {R}')
print(f'Ring generators: {R.gens()}')
print(f'Response generators (n_tilde={ft._n_tilde}): {R.gens()[:ft._n_tilde]}')
print(f'Physical generators: {R.gens()[ft._n_tilde:]}')
print()

passed = ft.sanity_check()
print()
ft.summary()

Taylor order: 4
Polynomial ring: Multivariate Polynomial Ring in nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2 over Symbolic Ring
Ring generators: (nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2)
Response generators (n_tilde=4): (nt1, nt2, vt1, vt2)
Physical generators: (dn1, dn2, dv1, dv2)

=== Sanity checks ===
  [PASS]  (n_tilde=0, n_phys=0)  constant term
  [PASS]  (n_tilde=1, n_phys=0)  tadpole — must vanish at MF saddle
  [PASS]  (n_tilde=0, n_phys=1)  linear physical-only — must vanish at EOM

=== Action sectors ===
  (n_tilde=1, n_phys=1)  [free action]:


<IPython.core.display.Math object>

  (n_tilde=1, n_phys=2)  [vertex (order 3)]:


<IPython.core.display.Math object>

  (n_tilde=2, n_phys=0)  [noise kernel]:


...

<IPython.core.display.Math object>

  (n_tilde=4, n_phys=1)  [vertex (order 5)]:


<IPython.core.display.Math object>

  (n_tilde=4, n_phys=2)  [vertex (order 6)]:


<IPython.core.display.Math object>

### 1.2 Propagator

Extract the kernel matrix $\mathbf{K}$ from the free action, Fourier transform, and invert.

In [ ]:
import signal

S_free = ft.free_action()
ring_gen_names = [str(g) for g in R.gens()]

# Use ring variable ordering (must match build_field_index_map)
resp_names = ring_gen_names[:ft._n_tilde]
phys_names = ring_gen_names[ft._n_tilde:]

pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

nf = len(resp_names)
K_data = [[SR(0)] * nf for _ in range(nf)]
for exp_vec, coeff in S_free.dict().items():
    row = col = None
    for idx in range(len(ring_gen_names)):
        if exp_vec[idx] > 0:
            if idx in pos_to_row: row = pos_to_row[idx]
            if idx in pos_to_col: col = pos_to_col[idx]
    if row is not None and col is not None:
        K_data[row][col] += SR(coeff)

K_mat = matrix(SR, K_data)

# Convert to kernel form
Dt       = ns.Dt
delta_D  = ns.delta_D
delta_Dp = ns.delta_Dp

def _to_kernel(c):
    c = SR(c)
    if c.has(delta_D) or c.has(ns.g):
        return c
    p0   = c.subs({Dt: 0})
    rest = (c - p0).subs({Dt: delta_Dp})
    return p0 * delta_D + rest

K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

resp_sr  = [ns.vt[i] for i in ns.pop] + [ns.nt[i] for i in ns.pop]
phys_sr  = [ns.dv[i] for i in ns.pop] + [ns.dn[i] for i in ns.pop]

print('Field ordering:')
display(Math(r'\tilde{\mathbf{a}} = ' + latex(vector(resp_sr))
             + r',\qquad \mathbf{a} = ' + latex(vector(phys_sr))))
print()
print('Kernel matrix K (time domain):')
display(Math(r'\mathbf{K} = ' + latex(K_ker)))

# Fourier transform
t_var = SR.var('t')
omega = SR.var('omega', latex_name=r'\omega')

time_domain = {
    delta_D:  dirac_delta(t_var),
    delta_Dp: diff(dirac_delta(t_var), t_var),
    ns.g:     function('g')(t_var),
}

K_ft_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        c = K_ker[i, j]
        if not c.is_zero():
            K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

K_ft = matrix(SR, K_ft_data)
print('Fourier-domain kernel:')
display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

# Propagator inverse
G_ft = K_ft.inverse().apply_map(lambda e: e.factor())
G_ft_explicit = True
print('Propagator:')
display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))

# Adjugate, determinant, poles
adj_ft  = K_ft.adjugate()
D_omega = K_ft.det().expand()
D_prime = diff(D_omega, omega)

pole_eqs = solve(D_omega == 0, omega)
pole_vals = [eq.rhs() if hasattr(eq, 'rhs') else eq for eq in pole_eqs]

print(f'\ndet(K̂) = {D_omega}')
print(f'Poles ({len(pole_vals)}):')
for pk, p in enumerate(pole_vals):
    display(Math(r'\omega_{' + str(pk+1) + '} = ' + latex(p)))

# Residue matrices
C_mats = []
for omega_k in pole_vals:
    C_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            n_ij = adj_ft[i, j]
            if not n_ij.is_zero():
                num = n_ij.subs({omega: omega_k})
                den = D_prime.subs({omega: omega_k})
                C_data[i][j] = (I * num / den).factor()
    C_mats.append(matrix(SR, C_data))

print('Residue matrices:')
for pk, C in enumerate(C_mats):
    display(Math(r'\mathbf{C}_{' + str(pk+1) + r'} = ' + latex(C)))

# Time-domain propagator (smooth part)
G_t = sum(C_mats[pk] * exp(I * pole_vals[pk] * t_var) for pk in range(len(pole_vals)))
G_t = G_t.apply_map(lambda e: e.simplify_full())

# Delta-function coefficients: D[i,j] = lim_{ω→∞} G_ft[i,j](ω)
# Nonzero whenever the propagator entry has an instantaneous (δ) component.
from sage.all import CDF as _CDF
D_delta_data = [[SR(0)] * nf for _ in range(nf)]
_omega_big = 1e12
for i in range(nf):
    for j in range(nf):
        entry = G_ft[i, j]
        try:
            v1 = complex(_CDF(entry.subs({omega: _omega_big})))
            v2 = complex(_CDF(entry.subs({omega: 2 * _omega_big})))
            if abs(v1) > 1e-9 and abs(v1 - v2) < 1e-6 * abs(v1):
                D_delta_data[i][j] = SR(round(v1.real, 6))
        except Exception:
            pass
D_delta = matrix(SR, D_delta_data)

print('Delta-function coefficient matrix:')
display(Math(r'\mathbf{D} = ' + latex(D_delta)))

print()
print('Full retarded propagator:')
display(Math(
    r'\mathbf{G}^R(t) = \mathbf{D}\,\delta(t)'
    r' \;+\; \Theta(t) \sum_{k=1}^{' + str(len(pole_vals)) + r'}'
    r' \mathbf{C}_k \, e^{i\omega_k t}'
))

---
## 2. Prediagram Enumeration

Enumerate all trees, topologies, and prediagrams for the $k$-point function
at each loop order $\ell = 0, \ldots, \ell_{\max}$.

In [5]:
from sage.plot.plot import graphics_array

def plot_prediagrams(pds, title_prefix=''):
    """Plot prediagrams with colored vertices."""
    if not pds:
        print('  (none)')
        return
    plots = []
    for i, (D, G, leaves, internal) in enumerate(pds):
        leaves_set = set(leaves)
        color_map = {}
        for v in D.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            elif D.in_degree(v) == 0:
                color_map.setdefault('red', []).append(v)
            else:
                color_map.setdefault('lightblue', []).append(v)
        plots.append(D.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

def plot_topologies(topos, title_prefix=''):
    """Plot undirected topologies."""
    if not topos:
        print('  (none)')
        return
    plots = []
    for i, (G, leaves, internal) in enumerate(topos):
        leaves_set = set(leaves)
        color_map = {}
        for v in G.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            else:
                color_map.setdefault('lightgray', []).append(v)
        plots.append(G.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

print('Plot helpers defined.')

Plot helpers defined.


In [ ]:
# ── Pipeline cache (depends on model + k) ──────────────────────
# Model name + external fields sanitized for filesystem use
_model_tag = HAWKES_MODEL['name'].replace(' ', '_').lower()
_ext_tag = '_'.join(f'{f[0]}{f[1]}' for f in external_fields)
cache_dir = f'saved_results/{_model_tag}_k{k}_{_ext_tag}'
cache = PipelineCache(cache_dir)
if USE_CACHE:
    print(f'Cache: {cache}')
    for e in cache.list_cached():
        print(f'  {e["key"]}  (saved {e["saved_at"][:19]})')
else:
    print('Cache disabled — all stages will be recomputed.')

# ── Enumerate prediagrams for each loop order ──────────────────
pds_by_ell = {}    # ell -> list of prediagrams
counts_by_ell = {} # ell -> counts dict

for ell in range(max_ell + 1):
    def _enumerate(ell=ell):
        t, tp, pd, c = enumerate_all(k, ell, verbose=False)
        return (pd, c)

    pds, counts = cache.get_or_compute(
        'prediagrams', _enumerate, k=k, loop_order=ell
    ) if USE_CACHE else _enumerate()

    pds_by_ell[ell] = pds
    counts_by_ell[ell] = counts

    print(f'ell={ell}:  {counts["n_trees"]} trees,  '
          f'{counts["n_topologies"]} topologies,  '
          f'{counts["n_prediagrams"]} prediagrams')
    plot_prediagrams(pds, title_prefix=f'ell={ell} PD ')

---
## 3. Vertex Extraction

Extract typed vertices (`VertexType`, `SourceType`) from the expanded action, based on the maximum vertex degree required by the prediagrams.

In [ ]:
from msrjd.enumeration.degree_scan import max_vertex_degree, scan_source_vertices

all_pds = [pd for pds in pds_by_ell.values() for pd in pds]
max_deg = max_vertex_degree(all_pds)
src_degs = scan_source_vertices(all_pds)
print(f'Max vertex degree across all prediagrams: {max_deg}')
print(f'Source vertex out-degrees needed: {src_degs}')
print(f'Current Taylor order: {ft.taylor_order}')
print(f'Sufficient: {ft.taylor_order >= max_deg}')
print()

vtypes = extract_vertex_types(ft)
stypes = extract_source_types(ft)

print(f'── Interaction vertex types ({len(vtypes)}) ──')
for i, vt in enumerate(vtypes):
    print(f'  V{i+1}: bigrade={vt.bigrade}, in_deg={vt.in_degree}, out_deg={vt.out_degree}')
    print(f'        resp_legs={vt.response_legs}, phys_legs={vt.physical_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(vt.coefficient)}'))

print(f'\n── Source types ({len(stypes)}) ──')
for i, st in enumerate(stypes):
    print(f'  S{i+1}: bigrade={st.bigrade}, out_deg={st.out_degree}')
    print(f'        resp_legs={st.response_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(st.coefficient)}'))

int_degs, src_degs_avail = available_degrees(vtypes, stypes)
print(f'\nAvailable interaction degrees (in, out): {sorted(int_degs)}')
print(f'Available source out-degrees: {sorted(src_degs_avail)}')

---
## 4. Prediagram Filtering

Remove prediagrams whose vertex degree signatures don't match any available vertex or source type.

In [ ]:
kept_by_ell = {}  # ell -> list of kept prediagrams

for ell in range(max_ell + 1):
    def _filter(ell=ell):
        kept, disc = filter_prediagrams(pds_by_ell[ell], vtypes, stypes)
        return (kept, disc)

    kept, disc = cache.get_or_compute(
        'filtered', _filter, k=k, loop_order=ell
    ) if USE_CACHE else _filter()

    kept_by_ell[ell] = kept
    print(f'ell={ell}:  {len(pds_by_ell[ell])} prediagrams -> '
          f'{len(kept)} kept, {disc} discarded')
    plot_prediagrams(kept, title_prefix=f'ell={ell} PD ')

---
## 5. Unique Typed Diagrams

For each surviving prediagram, enumerate all valid field-type assignments,
filter for causality, and deduplicate by isomorphism to obtain the set of
**unique Feynman diagrams** $\Gamma$.  Only these are cached — intermediate
typed diagrams are not stored.

In [ ]:
# External fields are specified in the Configuration cell above.
print(f'External fields (k={k}): {external_fields}')

# Build field index maps
ring_var_names_list = list(ns._ring_var_names)
n_tilde = ft._n_tilde
resp_idx, phys_idx = build_field_index_map(ring_var_names_list, n_tilde)

# Validate that each external field exists in the physical field index
for field in external_fields:
    assert field in phys_idx, (
        f'External field {field} not found in physical fields. '
        f'Available: {sorted(phys_idx.keys())}'
    )

print('\nResponse field index map:')
for leg, idx in sorted(resp_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> row {idx}')
print()
print('Physical field index map:')
for leg, idx in sorted(phys_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> col {idx}')

In [ ]:
unique_by_ell = {}  # ell -> list of unique TypedDiagram

for ell in range(max_ell + 1):
    def _unique(ell=ell):
        typed = enumerate_all_typed(
            kept_by_ell[ell], external_fields, vtypes, stypes,
            G_ft, resp_idx, phys_idx
        )
        causal, n_disc, _ = filter_causal(typed)
        unique = deduplicate_typed_diagrams(causal)
        print(f'  ell={ell}: {len(kept_by_ell[ell])} prediagrams -> {len(typed)} typed'
              f' -> {len(causal)} causal -> {len(unique)} unique')
        return unique

    unique_by_ell[ell] = cache.get_or_compute(
        'unique_typed', _unique, k=k, loop_order=ell
    ) if USE_CACHE else _unique()

    print(f'ell={ell}: {len(unique_by_ell[ell])} unique diagrams')

# Convenience aliases used downstream
unique_tree = unique_by_ell.get(0, [])
all_unique = [td for ell in range(max_ell + 1) for td in unique_by_ell.get(ell, [])]
print(f'\nTotal unique diagrams across all loop orders: {len(all_unique)}')

---
## 6. Diagram Details & Combinatorial Factor $M(\Gamma)$

For each unique diagram $\Gamma$, display the vertex assignments, edge types,
and the **combinatorial factor** $M(\Gamma)$ together with the scalar prefactor.

The vertex coefficients shown include the $(-1)$ from the partition function
convention $Z = \int \exp(-S)$.

In [ ]:
from msrjd.diagrams.symmetry import classify_coefficient_factors

# Get time-dependence metadata from the model
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure', {
    'temporal_type': 'white', 'amplitude_params': []
})
print(f'Noise temporal type: {noise_structure.get("temporal_type", "white")}')
print(f'Time-dependent params: {time_dep_params if time_dep_params else "(none -- stationary)"}')
print()


def show_unique_diagram(td, idx, winfo):
    """Display a unique diagram with M(Gamma) and weight structure."""
    M = winfo['M']
    print(f'\n{"="*60}')
    print(f'Unique Diagram {idx}    M = {M}')
    print(f'{"="*60}')

    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} <- {field[0]}{field[1]}')

    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            eff_coeff = -SR(vtype.coefficient)
            display(Math(f'\\quad (-1)\\cdot c_{{v_{v}}} = {latex(eff_coeff)}'))

    print('  Edges (propagator assignments):')
    for edge_key in sorted(td.edge_types.keys()):
        resp_leg, phys_leg = td.edge_types[edge_key]
        u, v = edge_key[0], edge_key[1]
        ri, pi = td.propagator_indices.get(edge_key, ('?', '?'))
        print(f'    {u} -> {v}:  {resp_leg[0]}{resp_leg[1]} -> '
              f'{phys_leg[0]}{phys_leg[1]}  [G_hat[{ri},{pi}]]')

    sp = winfo['scalar_prefactor']
    display(Math(
        r'\text{Scalar prefactor: }\;' + latex(sp)
    ))


# ── Display all unique diagrams by loop order ──
for ell in range(max_ell + 1):
    diagrams = unique_by_ell.get(ell, [])
    ell_label = 'TREE-LEVEL' if ell == 0 else f'{ell}-LOOP'
    print('='*60)
    print(f'{ell_label} UNIQUE DIAGRAMS ({len(diagrams)})')
    print('='*60)

    for i, td in enumerate(diagrams, 1):
        winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
        label = f'Tree-{i}' if ell == 0 else f'{ell}L-{i}'
        show_unique_diagram(td, label, winfo)

---
## 7. Symbolic Integration (Stationary Case)

For each unique diagram $\Gamma$, construct the **unevaluated frequency-domain
integral** that gives the time-domain contribution $C_\Gamma(t_1, t_2)$.

### Procedure

1. **Assign** a frequency variable $\omega_{e_i}$ to each directed edge.

2. **Solve conservation** at every internal vertex (interaction and source alike):
   $$\delta\!\Big(\sum_{\text{in}} \omega_e - \sum_{\text{out}} \omega_e\Big)$$
   This expresses internal edge frequencies in terms of external ones.
   If the diagram has $\ell$ loops, $\ell$ internal frequencies remain free.

3. **Build the integrand**: for each edge, substitute the resolved frequency
   into the propagator entry $\hat{G}_{i,j}(\omega_e)$.  Each external leg
   contributes an exponential $e^{\pm i\omega t}$ from the inverse Fourier
   transform (sign from leaf directionality: tail $\to e^{-i\omega t}$,
   head $\to e^{+i\omega t}$).

4. **Display** the full unevaluated integral:
   $$C_\Gamma(t_1, t_2) = \text{scalar\_pf}
   \;\prod_j \int\!\frac{d\omega_j}{2\pi}\;
   \Bigl[\prod_e \hat{G}_{i_e,j_e}(\omega_e)\Bigr]
   \Bigl[\prod_{\text{ext}} e^{\pm i\omega t}\Bigr]$$

In [ ]:
from msrjd.integration.symbolic import (
    check_propagator_available,
    build_integrand_stationary,
    extract_propagator_factors,
    canonicalize_prop_factors,
    loop_kernel_signature,
    group_diagrams_by_kernel,
)

# ── Propagator data dict (assembled from Section 1.2) ──
omega = SR.var('omega', latex_name=r'\omega')

propagator_data = {
    'G_ft': G_ft,
    'adj_ft': adj_ft,
    'D_omega': D_omega,
    'G_ft_explicit': True,
    'pole_vals': pole_vals,
    'C_mats': C_mats,
    'nf': nf,
}

prop_mode = check_propagator_available(propagator_data)
print(f'Propagator mode: {prop_mode}')
print(f'Poles: {pole_vals}')
print()


def _diagram_label(td):
    """Return a human-readable label like 'Tree-3' or '1L-17'."""
    for ell in range(max_ell + 1):
        diagrams = unique_by_ell.get(ell, [])
        if td in diagrams:
            idx = diagrams.index(td) + 1
            return f'Tree-{idx}' if ell == 0 else f'{ell}L-{idx}'
    return '??'


def show_integral(td, label, propagator_data, omega_sym):
    """
    Build and display the unevaluated integral for a typed diagram.
    Returns the integrand result dict.
    """
    ir = build_integrand_stationary(
        td, propagator_data, k=k,
        omega_symbol=omega_sym,
        time_dep_params=HAWKES_MODEL.get('time_dependent_parameters', []),
        noise_structure=HAWKES_MODEL.get('noise_structure'),
    )

    prefactor = ir['scalar_prefactor']
    full_integrand = ir['full_integrand']
    ext_freqs = ir['ext_freqs']
    free_freqs = ir['free_freqs']
    ell = ir['loop_number']
    overall = ir['overall_conservation']

    print(f'\n{"="*60}')
    print(f'{label}   (ell = {ell})')
    print(f'{"="*60}')

    print(f'  Free (loop) frequencies: {free_freqs if free_freqs else "(none)"}')
    print(f'  Independent external frequencies: {ext_freqs}')

    if overall is not None:
        display(Math(
            r'  \text{Overall conservation: }\;'
            + latex(overall) + r' = 0'
        ))

    int_vars = list(free_freqs) + list(ext_freqs)
    integrals_tex = ''
    for v in int_vars:
        integrals_tex += r'\int\!\frac{d' + latex(v) + r'}{2\pi}\;'

    pf_tex = latex(prefactor)
    integrand_tex = latex(full_integrand)

    display(Math(
        r'C_{\Gamma}(t_1, t_2) = '
        + pf_tex + r'\;'
        + integrals_tex
        + integrand_tex
    ))

    integrand_only = ir['integrand']
    display(Math(
        r'\text{Propagator product: }\;'
        + latex(integrand_only)
    ))

    exp_factor = (full_integrand / integrand_only)
    try:
        exp_factor = exp_factor.simplify()
    except Exception:
        pass
    display(Math(
        r'\text{Exponential factor: }\;'
        + latex(exp_factor)
    ))

    return ir


# ═══════════════════════════════════════════════════════════════
# Show integrals for each loop order
# ═══════════════════════════════════════════════════════════════
ir_by_ell = {}  # ell -> list of integrand results

for ell in range(max_ell + 1):
    diagrams = unique_by_ell.get(ell, [])
    ell_label = 'TREE-LEVEL' if ell == 0 else f'{ell}-LOOP'

    print('='*60)
    print(f'{ell_label} INTEGRALS ({len(diagrams)})')
    print('='*60)

    ir_list = []
    for i, td in enumerate(diagrams, 1):
        label = f'Tree-{i}' if ell == 0 else f'{ell}L-{i}'
        ir_i = show_integral(td, label, propagator_data, omega)
        ir_list.append(ir_i)
    ir_by_ell[ell] = ir_list

### 7.2 Unique loop kernels

Group all diagrams by their **loop kernel** -- the product of propagator entries
$\prod_e \hat{G}_{i_e,j_e}(\omega_e)$ with the same frequency routing.

Diagrams sharing a loop kernel differ only in their scalar prefactors (vertex
coefficients, combinatorial factors).  These prefactors simply add, so we only
need to numerically integrate each unique kernel once.

For each unique kernel, we show:
- Which diagrams it came from
- The combined scalar prefactor
- The propagator factors (opaque representation)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Group ALL diagrams by loop kernel
# ═══════════════════════════════════════════════════════════════
from msrjd.integration.symbolic import _factor_depends_on

kernel_groups = group_diagrams_by_kernel(
    all_unique, propagator_data, k=k,
    omega_symbol=omega,
    time_dep_params=HAWKES_MODEL.get('time_dependent_parameters', []),
    noise_structure=HAWKES_MODEL.get('noise_structure'),
)

print(f'Total unique diagrams: {len(all_unique)}')
print(f'Unique loop kernels:   {len(kernel_groups)}')
print(f'Numerical integrations saved: {len(all_unique) - len(kernel_groups)}')
print()


def _factor_to_latex(f):
    """Render a single opaque propagator factor as LaTeX."""
    if f[0] == 'prop':
        _, ri, pi, omega_expr = f
        return (r'\hat{G}_{' + str(ri) + ',' + str(pi) + r'}('
                + latex(omega_expr) + ')')
    elif f[0] == 'noise':
        return r'\hat{\kappa}(' + latex(f[1]) + ')'
    return str(f)


for g_idx, g in enumerate(kernel_groups, 1):
    ell = g['loop_number']
    n = g['n_diagrams']
    combined_pf = g['combined_prefactor']
    ir = g['representative_ir']

    print(f'{"="*60}')
    print(f'Kernel {g_idx}   (ell = {ell}, {n} diagram{"s" if n > 1 else ""})')
    print(f'{"="*60}')

    for j, td in enumerate(g['diagrams']):
        pf_j = g['individual_prefactors'][j]
        print(f'  {_diagram_label(td)}:  prefactor = {pf_j}')

    display(Math(
        r'\text{Combined prefactor: }\;'
        + latex(combined_pf)
    ))

    pf_list = g['prop_factors']
    loop_vars_canonical = [SR.var(f'L_{i}') for i in range(ell)]

    if ell > 0:
        loop_factors = [f for f in pf_list
                        if _factor_depends_on(f, loop_vars_canonical)]
        ext_factors = [f for f in pf_list
                       if not _factor_depends_on(f, loop_vars_canonical)]
    else:
        loop_factors = []
        ext_factors = list(pf_list)

    ext_tex = (r' \cdot '.join(_factor_to_latex(f) for f in ext_factors)
               if ext_factors else '')
    loop_tex = (r' \cdot '.join(_factor_to_latex(f) for f in loop_factors)
                if loop_factors else '')

    if ell == 0:
        if ext_tex:
            display(Math(r'\text{Propagator product: }\;' + ext_tex))
    else:
        loop_var_tex = r' \cdot '.join(
            r'\frac{d' + latex(v) + r'}{2\pi}'
            for v in loop_vars_canonical
        )
        integral_tex = r'\int_{-\infty}^{\infty} ' + loop_var_tex
        if loop_tex:
            integral_tex += r'\; ' + loop_tex

        if ext_tex:
            full_tex = ext_tex + r' \;\times\; ' + integral_tex
        else:
            full_tex = integral_tex

        display(Math(r'\text{Loop integral: }\;' + full_tex))

        loop_vars_tex = ', '.join(latex(v) for v in ir['free_freqs'])
        ext_vars_tex = ', '.join(latex(v) for v in ir['ext_freqs'])
        print(f'  Loop variables: {{{loop_vars_tex}}}')
        print(f'  External variables: {{{ext_vars_tex}}}')

    print()

### 7.3 Unique loop integrands

Section 7.2 groups diagrams by their **full** kernel signature
(external factors + loop integrand).  But for numerical integration
we only need to evaluate each **distinct loop integrand** once --
diagrams that share the same loop integrand but differ in their
external propagator factors still require only one numerical
integration.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.3  Unique loop integrands
# ═══════════════════════════════════════════════════════════════
from collections import defaultdict as _defaultdict
from msrjd.integration.symbolic import loop_only_signature

# ── Group kernel_groups by their loop-only signature ─────────
loop_integrand_groups = _defaultdict(list)  # loop_sig -> list of kernel group dicts

for g in kernel_groups:
    if g['loop_number'] == 0:
        continue
    lsig = loop_only_signature(g['signature'])
    loop_integrand_groups[lsig].append(g)

n_tree = sum(1 for g in kernel_groups if g['loop_number'] == 0)
n_full_loop_kernels = sum(1 for g in kernel_groups if g['loop_number'] > 0)
n_unique_loop_integrands = len(loop_integrand_groups)
n_loop_diagrams = sum(
    g['n_diagrams'] for g in kernel_groups if g['loop_number'] > 0
)

print(f'Tree-level kernels (no integration needed):      {n_tree}')
print(f'Full loop kernels (ext + loop signature):        {n_full_loop_kernels}')
print(f'Unique loop integrands (loop signature only):    {n_unique_loop_integrands}')
print(f'Total loop diagrams covered:                     {n_loop_diagrams}')
print(f'Numerical integrations saved:                    '
      f'{n_loop_diagrams - n_unique_loop_integrands}')
print()

for li, (lsig, groups) in enumerate(loop_integrand_groups.items(), 1):
    ell = groups[0]['loop_number']
    all_labels = []
    total_diagrams = 0
    for g in groups:
        total_diagrams += g['n_diagrams']
        for td in g['diagrams']:
            all_labels.append(_diagram_label(td))

    pf_list = groups[0]['prop_factors']
    loop_vars_canonical = [SR.var(f'L_{i}') for i in range(ell)]

    lf = [f for f in pf_list if _factor_depends_on(f, loop_vars_canonical)]

    loop_var_tex = r' \cdot '.join(
        r'\frac{d' + latex(v) + r'}{2\pi}'
        for v in loop_vars_canonical
    )
    integrand_body = r' \cdot '.join(_factor_to_latex(f) for f in lf) if lf else '1'
    integrand_tex = (r'\int_{-\infty}^{\infty} '
                     + loop_var_tex + r'\; ' + integrand_body)

    label_str = ', '.join(all_labels)
    if len(all_labels) > 6:
        label_str = ', '.join(all_labels[:6]) + f', ... ({total_diagrams} total)'

    print(f'── Loop integrand {li}/{n_unique_loop_integrands}  '
          f'(ell={ell}, {total_diagrams} diagram{"s" if total_diagrams>1 else ""}: '
          + label_str + ')')
    display(Math(integrand_tex))
    print()

### 7.4 Numerical evaluation -- spectral grid + inverse FFT

For each unique loop kernel, numerically evaluate the loop integral
on a spectral grid, then combine with the tree-level contributions
and IFFT to the time domain.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.4  Numerical evaluation — adaptive by k
# ═══════════════════════════════════════════════════════════════
import numpy as np
from scipy.optimize import fsolve
from sage.all import numerical_integral, CC, fast_callable, CDF, diff

# ═══════════════════════════════════════════════════════════════
# Fundamental parameters (model-agnostic: user specifies these)
# ═══════════════════════════════════════════════════════════════
fundamental = {
    'E':   [1.0, 0.5],                        # external drive per population
    'w':   [[0.3, 0.5], [0.1, 0.3]],         # synaptic weights w_ij
    'tau': 10.0,                               # membrane time constant
}

npop = len(ns.pop)

print('Fundamental parameters:')
for key, val in fundamental.items():
    print(f'  {key} = {val}')

# ═══════════════════════════════════════════════════════════════
# Solve mean-field equations from the model
# ═══════════════════════════════════════════════════════════════
# Read phi_concrete from model and differentiate symbolically
v_sym = SR.var('_v_mf_')
taylor_order = ft.taylor_order

# Build substitution dict: map model parameter symbols to fundamental values.
# This handles any model (linear phi has no 'a', quadratic has 'a', etc.)
_param_subs = {}
for pspec in HAWKES_MODEL.get('parameters', []):
    pname = pspec['name']
    if pname in fundamental:
        if pspec.get('indexed', False):
            for i in ns.pop:
                sym = getattr(ns, pname)[i]
                _param_subs[sym] = fundamental[pname][i]
        else:
            sym = getattr(ns, pname)
            _param_subs[sym] = fundamental[pname]

phi_derivs = {}  # phi_derivs[i][dk] = k-th derivative of phi_i(v), symbolic
for i in ns.pop:
    phi_expr = HAWKES_MODEL['phi_concrete'](ns, i, v_sym)
    phi_derivs[i] = {}
    for dk in range(taylor_order + 1):
        if dk == 0:
            phi_derivs[i][dk] = phi_expr
        else:
            phi_derivs[i][dk] = diff(phi_expr, v_sym, dk)

# Build numerical phi callable from the symbolic expression
def phi_num(i, v_val):
    """Evaluate phi_i(v) numerically."""
    return float(phi_derivs[i][0].subs(_param_subs).subs({v_sym: v_val}))

# Solve MF self-consistency: n*_i = phi_i(E_i + sum_j w_ij * n*_j)
def mf_residual(nstar_vec):
    residuals = []
    for i in ns.pop:
        v_star_i = fundamental['E'][i] + sum(
            fundamental['w'][i][j] * nstar_vec[j] for j in ns.pop
        )
        residuals.append(nstar_vec[i] - phi_num(i, v_star_i))
    return residuals

# Initial guess: small positive rates
nstar_guess = [1.0] * npop
nstar_sol = fsolve(mf_residual, nstar_guess, full_output=True)
nstar_vals = [float(x) for x in nstar_sol[0]]
info = nstar_sol[1]

# Compute v* and phi derivatives at the fixed point
vstar_vals = []
phi_deriv_vals = {}  # phi_deriv_vals[(dk, i)] = d^k phi_i / dv^k |_{v=v*}
for i in ns.pop:
    v_star_i = fundamental['E'][i] + sum(
        fundamental['w'][i][j] * nstar_vals[j] for j in ns.pop
    )
    vstar_vals.append(v_star_i)
    for dk in range(taylor_order + 1):
        phi_deriv_vals[(dk, i)] = float(
            phi_derivs[i][dk].subs(_param_subs).subs({v_sym: v_star_i})
        )

# Sanity check: phi_i(v*_i) should equal n*_i
print(f'\nMean-field solution:')
for i in ns.pop:
    phi_at_vstar = phi_deriv_vals[(0, i)]
    print(f'  pop {i+1}:  v* = {vstar_vals[i]:.6f},  '
          f'n* = {nstar_vals[i]:.6f},  '
          f'phi(v*) = {phi_at_vstar:.6f}  '
          f'{"OK" if abs(phi_at_vstar - nstar_vals[i]) < 1e-10 else "MISMATCH!"}')

print(f'\nDerived phi derivatives:')
for i in ns.pop:
    derivs_str = ', '.join(
        f"phi{dk}_{i+1} = {phi_deriv_vals[(dk, i)]:.6f}"
        for dk in range(1, taylor_order + 1)
        if (dk, i) in phi_deriv_vals
    )
    print(f'  pop {i+1}:  {derivs_str}')

# ═══════════════════════════════════════════════════════════════
# Assemble num_params for symbolic substitution
# ═══════════════════════════════════════════════════════════════
num_params = {}

# Weights
for i in ns.pop:
    for j in ns.pop:
        num_params[SR.var(f'w{i+1}{j+1}')] = fundamental['w'][i][j]

# Timescale
num_params[SR.var('tau')] = fundamental['tau']

# Derived: nstar
for i in ns.pop:
    num_params[ns.nstar[i]] = float(nstar_vals[i])

# Derived: phi derivatives (phi1, phi2, ..., up to taylor_order)
for i in ns.pop:
    for dk in range(1, taylor_order + 1):
        sym = SR.var(f'phi{dk}_{i+1}')
        if (dk, i) in phi_deriv_vals:
            num_params[sym] = phi_deriv_vals[(dk, i)]

print(f'\nFull num_params:')
for sym, val in num_params.items():
    print(f'  {sym} = {val}')

# Check pole stability
print(f'\nPoles (numerical):')
for p_idx, p in enumerate(pole_vals):
    p_num = complex(CC(p.subs(num_params)))
    print(f'  pole {p_idx+1}: {p_num:.6f}   (Im = {p_num.imag:.4f})')


# ═══════════════════════════════════════════════════════════════
# Evaluation helpers
# ═══════════════════════════════════════════════════════════════

def _find_integrand_vars(ir, num_params):
    """
    Substitute num_params into ir['integrand'] and identify which
    omega variables remain.  Returns (integrand_num, ext_vars, loop_vars).
    ext_vars/loop_vars are lists (possibly empty).
    """
    integrand_num = ir['integrand'].subs(num_params)
    remaining = sorted(integrand_num.variables(), key=str)

    ext_set = set(ir['ext_freqs'])
    free_set = set(ir['free_freqs'])

    ext_vars = [v for v in remaining if v in ext_set]
    loop_vars = [v for v in remaining if v in free_set]
    other = [v for v in remaining if v not in ext_set and v not in free_set]
    loop_vars += other  # anything unclassified goes to loop

    return integrand_num, ext_vars, loop_vars


def _scalar_loop_integral(ir, num_params, Omega_max=50, n_quad=1000):
    """
    Compute a pure scalar loop integral (no external frequencies).
    Returns a single complex number.
    """
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_vars, loop_vars = _find_integrand_vars(ir, num_params)

    if not loop_vars:
        # No integration needed — just a constant (tree-level for k=1)
        return prefactor * complex(CDF(integrand_num))

    omega_loop = loop_vars[0]
    f_fast = fast_callable(integrand_num, vars=[omega_loop], domain=CDF)
    Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
    dOmega = Omega_pts[1] - Omega_pts[0]
    vals = np.array([complex(f_fast(Om)) for Om in Omega_pts])
    result = np.trapz(vals, dx=dOmega) / (2 * np.pi)
    return prefactor * result


def spectrum_tree(ir, num_params, omega_grid):
    """Evaluate tree-level spectrum Chat on grid.
    
    For k=2 (1 ext freq): returns 1D array Chat(omega).
    For k>=3 (multiple ext freqs): returns nD array Chat(omega_0, omega_1, ...).
    """
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_vars, loop_vars = _find_integrand_vars(ir, num_params)

    if not ext_vars:
        val = prefactor * complex(CDF(integrand_num))
        if isinstance(omega_grid, np.ndarray) and omega_grid.ndim == 1:
            return np.full(len(omega_grid), val, dtype=complex)
        return val

    n_ext = len(ext_vars)
    f_fast = fast_callable(integrand_num, vars=ext_vars, domain=CDF)

    if n_ext == 1:
        Chat = np.array([complex(f_fast(w)) for w in omega_grid])
    elif n_ext == 2:
        N = len(omega_grid)
        Chat = np.zeros((N, N), dtype=complex)
        for i, w0 in enumerate(omega_grid):
            for j, w1 in enumerate(omega_grid):
                Chat[i, j] = complex(f_fast(w0, w1))
    else:
        raise NotImplementedError(
            f'spectrum_tree: {n_ext} external frequencies not yet supported. '
            f'Need {n_ext}D grid evaluation.')

    return prefactor * Chat


def spectrum_one_loop(ir, num_params, omega_grid, Omega_max=50, n_quad=1000):
    """
    Evaluate one-loop spectrum Chat(omega) on a grid of external frequency.
    For each external omega, numerically integrate the loop frequency.
    """
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_vars, loop_vars = _find_integrand_vars(ir, num_params)

    if not loop_vars:
        print('  1-loop: no loop variable found — treating as tree')
        return spectrum_tree(ir, num_params, omega_grid)

    omega_loop = loop_vars[0]

    if not ext_vars:
        # Pure loop integral, no external freq.  Return scalar broadcast.
        f_fast = fast_callable(integrand_num, vars=[omega_loop], domain=CDF)
        Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
        dOmega = Omega_pts[1] - Omega_pts[0]
        vals = np.array([complex(f_fast(Om)) for Om in Omega_pts])
        scalar = prefactor * np.trapz(vals, dx=dOmega) / (2 * np.pi)
        return np.full(len(omega_grid), scalar, dtype=complex)

    ext_var = ext_vars[0]
    f_fast = fast_callable(integrand_num, vars=[ext_var, omega_loop], domain=CDF)
    Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
    dOmega = Omega_pts[1] - Omega_pts[0]

    Chat = np.zeros(len(omega_grid), dtype=complex)
    for i, w_ext in enumerate(omega_grid):
        vals = np.array([complex(f_fast(w_ext, Om)) for Om in Omega_pts])
        Chat[i] = np.trapz(vals, dx=dOmega) / (2 * np.pi)

    return prefactor * Chat


def inverse_fourier(Chat, omega_grid):
    """Inverse DFT: C(tau) where tau = t1 - t2.
    
    The MSR-JD integrand encodes C(t1-t2) via exp(+iw(t1-t2)).
    numpy ifft with e^{+iwt} gives C(t1-t2) directly.
    
    Handles 1D (k=2) and 2D (k=3) spectra.
    """
    N = len(omega_grid)
    Delta_omega = omega_grid[1] - omega_grid[0]
    tau_grid = np.fft.fftshift(np.fft.fftfreq(N, d=Delta_omega / (2 * np.pi)))
    scale = N * Delta_omega / (2 * np.pi)

    if Chat.ndim == 1:
        Chat_shifted = np.fft.ifftshift(Chat)
        C_tau = np.fft.fftshift(np.fft.ifft(Chat_shifted)) * scale
    elif Chat.ndim == 2:
        Chat_shifted = np.fft.ifftshift(Chat, axes=(0, 1))
        C_tau = np.fft.fftshift(np.fft.ifft2(Chat_shifted)) * scale**2
    else:
        raise NotImplementedError(f'inverse_fourier: {Chat.ndim}D not supported')

    return tau_grid, C_tau



def find_spectrum_poles(propagator_data, num_params):
    """Find all poles of the spectrum from the propagator poles.
    
    Poles of S(omega) come from det(K(omega))=0 and det(K(-omega))=0.
    """
    pole_syms = propagator_data['pole_vals']
    all_poles = []
    for p_sym in pole_syms:
        p_num = complex(CDF(p_sym.subs(num_params)))
        all_poles.append(p_num)
        all_poles.append(-p_num)
    return all_poles


def compute_numerical_residues(f_spectrum, poles, eps=1e-9):
    """Compute residues numerically via limit: Res(f,p) = lim (z-p)*f(z)."""
    residues = {}
    for pole in poles:
        z = pole + eps
        residues[pole] = (z - pole) * complex(f_spectrum(z))
    return residues


def ift_via_residues(f_spectrum, poles, tau_grid):
    """Compute C(tau) = (1/2pi) int S(omega) e^{+iw*tau} dw via residues.
    
    tau > 0: close upper -> C = +i * sum(upper residues * exp)
    tau < 0: close lower -> C = -i * sum(lower residues * exp)
    """
    residues = compute_numerical_residues(f_spectrum, poles)
    upper = [(p, r) for p, r in residues.items() if p.imag > 0]
    lower = [(p, r) for p, r in residues.items() if p.imag < 0]

    C = np.zeros(len(tau_grid), dtype=complex)
    for i, tau in enumerate(tau_grid):
        if tau > 0:
            C[i] = 1j * sum(r * np.exp(1j * p * tau) for p, r in upper)
        elif tau < 0:
            C[i] = -1j * sum(r * np.exp(1j * p * tau) for p, r in lower)
        else:
            val_u = sum(r for _, r in upper)
            val_l = sum(r for _, r in lower)
            C[i] = (1j * val_u - 1j * val_l) / 2
    return C


def evaluate_kernel_group(g, num_params, omega_grid=None,
                          Omega_max=50, n_quad=1000):
    """
    Evaluate a kernel group.
    - If omega_grid is None (k=1), computes a scalar loop integral.
    - Otherwise evaluates on the frequency grid.
    """
    ir = g['representative_ir']
    ell = g['loop_number']
    combined_pf = g['combined_prefactor']

    ir_eval = dict(ir)
    ir_eval['scalar_prefactor'] = combined_pf

    if omega_grid is None:
        # Scalar mode (k=1): just compute the loop integral
        return _scalar_loop_integral(ir_eval, num_params,
                                     Omega_max=Omega_max, n_quad=n_quad)

    if ell == 0:
        return spectrum_tree(ir_eval, num_params, omega_grid)
    else:
        return spectrum_one_loop(ir_eval, num_params, omega_grid,
                                 Omega_max=Omega_max, n_quad=n_quad)


# ═══════════════════════════════════════════════════════════════

def _build_factor_product(factors, propagator_data, omega_symbol, num_params):
    """Build a symbolic product from a list of canonical prop factors."""
    G_ft = propagator_data.get('G_ft')
    product = SR(1)
    for f in factors:
        if f[0] == 'prop':
            _, ri, pi, omega_expr = f
            if G_ft is not None:
                entry = SR(G_ft[ri, pi]).subs({omega_symbol: omega_expr})
            else:
                adj = propagator_data['adj_ft']
                det = propagator_data['D_omega']
                entry = (SR(adj[ri, pi]) / SR(det)).subs({omega_symbol: omega_expr})
            product *= entry
        elif f[0] == 'noise':
            pass  # white noise: no frequency-dependent factor
    return product.subs(num_params)


def _eval_product_on_grid(expr, ext_var, omega_grid):
    """Evaluate a symbolic expression on the omega grid."""
    if ext_var is None or not expr.has(ext_var):
        val = complex(CDF(expr))
        return np.full(len(omega_grid), val, dtype=complex)
    f_fast = fast_callable(expr, vars=[ext_var], domain=CDF)
    return np.array([complex(f_fast(w)) for w in omega_grid])


def _eval_loop_integral_on_grid(loop_integrand_expr, ext_var, loop_var,
                                omega_grid, num_params,
                                Omega_max=50, n_quad=1000):
    """
    Evaluate int loop_integrand(omega_ext, Omega) dOmega/(2pi)
    on the omega grid.  Returns array of shape (N_fft,).
    """
    if ext_var is not None and loop_var is not None:
        f_fast = fast_callable(loop_integrand_expr,
                               vars=[ext_var, loop_var], domain=CDF)
    elif loop_var is not None:
        f_fast = fast_callable(loop_integrand_expr,
                               vars=[loop_var], domain=CDF)
    else:
        # No loop variable — just a constant
        val = complex(CDF(loop_integrand_expr))
        return np.full(len(omega_grid), val, dtype=complex)

    Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
    dOmega = Omega_pts[1] - Omega_pts[0]

    if ext_var is not None:
        result = np.zeros(len(omega_grid), dtype=complex)
        for i, w_ext in enumerate(omega_grid):
            vals = np.array([complex(f_fast(w_ext, Om)) for Om in Omega_pts])
            result[i] = np.trapz(vals, dx=dOmega) / (2 * np.pi)
        return result
    else:
        vals = np.array([complex(f_fast(Om)) for Om in Omega_pts])
        scalar = np.trapz(vals, dx=dOmega) / (2 * np.pi)
        return np.full(len(omega_grid), scalar, dtype=complex)


# Dispatch by k
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

print(f'\n{"="*60}')
print(f'k = {k}  —  Evaluating {len(kernel_groups)} unique kernel(s)')
print(f'{"="*60}')

if k == 1:
    # ───────────────────────────────────────────────────────────
    # k = 1 :  Mean = mean-field + loop corrections
    # ───────────────────────────────────────────────────────────
    # The mean-field value n* comes from solving the classical rate
    # equations (saddle point of the action) — not from a diagram.
    # Diagrams compute fluctuation corrections on top of n*:
    #   <dn> = n* + delta<dn>_{1-loop} + delta<dn>_{2-loop} + ...

    # Compute unique loop integrals, sum per loop level
    from collections import defaultdict as _ddict
    correction_by_ell = _ddict(complex)

    if loop_integrand_groups:
        n_unique = len(loop_integrand_groups)
        n_total = sum(sum(g['n_diagrams'] for g in gs)
                      for gs in loop_integrand_groups.values())
        print(f'\n{n_unique} unique loop integrand(s), {n_total} diagram(s)')

        for li, (lsig, groups) in enumerate(loop_integrand_groups.items(), 1):
            ell = groups[0]['loop_number']
            total_diags = sum(g['n_diagrams'] for g in groups)

            # Evaluate the loop integral once with unit prefactor
            representative = groups[0]
            ir_unit = dict(representative['representative_ir'])
            ir_unit['scalar_prefactor'] = SR(1)
            raw_val = _scalar_loop_integral(ir_unit, num_params)

            # Sum (combined_prefactor × loop_integral) across groups
            val = sum(
                float(g['combined_prefactor'].subs(num_params)) * raw_val
                for g in groups
            )
            correction_by_ell[ell] += val

            print(f'  Integrand {li}/{n_unique} (ell={ell}, '
                  f'{total_diags} diagrams): {val:.6e}')

    # Report
    field_name, pop_idx = external_fields[0]
    field_label = f'{field_name}_{pop_idx}'
    nstar_sym = SR.var(f'nstar{pop_idx}')
    nstar_val = float(num_params.get(nstar_sym, 0))

    print(f'\n{"="*60}')
    print(f'k=1 Mean: <{field_label}>')
    print(f'{"="*60}')
    print(f'  {"Mean-field (n*)":25s} = {nstar_val: .6e}')

    loop_total = 0.0
    for ell in sorted(correction_by_ell):
        val = correction_by_ell[ell]
        label = f'{ell}-loop correction'
        re = val.real
        loop_total += re
        im_warn = ''
        if abs(val.imag) > 1e-12 * max(abs(val.real), 1e-30):
            im_warn = f'  [Im = {val.imag:.4e}]'
        print(f'  {label:25s} = {re: .6e}{im_warn}')

    total = nstar_val + loop_total
    print(f'  {chr(9472)*40}')
    print(f'  {"Total <" + field_label + ">":25s} = {total: .6e}')
    print(f'  (Loop / MF ratio:  {abs(loop_total/nstar_val):.2e})')

    # Stationary k=1: each loop order is a scalar — printed table
    # is the natural output.  (Nonstationary k=1 will produce <n>(t)
    # curves per loop order once that code path is implemented.)

else:
    # ───────────────────────────────────────────────────────────
    # k >= 2 :  Frequency grid + inverse FFT
    # ───────────────────────────────────────────────────────────
    n_ext = k - 1  # number of independent ext frequencies (stationary)

    if k >= 3:
        # Phase I pole / FFT IFT is unreliable at k>=3 (the nD
        # spectrum grid aliases badly and the residue path does
        # not generalize cleanly to multi-D time arguments). Skip
        # Phase I and let Phase J (Section 8) do the time-domain
        # integration instead. We still set up a τ grid that
        # downstream cells (Phase J evaluation and the simulation
        # comparison) can consume.
        print(f'\nk={k} >= 3: skipping Phase I pole/FFT IFT '
              f'(unreliable at this dimension).')
        print( '           Phase J (Section 8) will do the time-'
              'domain integration.')
        T_max = 80.0
        Delta_tau = 0.05
        tau_out = np.arange(-T_max, T_max + Delta_tau / 2, Delta_tau)
        tau_residue = tau_out.copy()
        C_tree_tau = None
        C_total_tau = None
        C_loop_tau = None
        C_tree_residue = None
        C_tree_tau_slices = None
        C_total_tau_slices = None
    else:
        if n_ext <= 1:
            T_max = 80.0
            Delta_tau = 0.05
        else:
            # k>=3: high resolution for clean nD IFT
            T_max = 80.0
            Delta_tau = 0.02
        N_fft = int(2 * T_max / Delta_tau)
        N_fft = int(2**np.ceil(np.log2(N_fft)))
        Delta_omega = 2 * np.pi / (N_fft * Delta_tau)
        omega_max = N_fft * Delta_omega / 2
        omega_grid = np.arange(-N_fft//2, N_fft//2) * Delta_omega
        mid = N_fft // 2

        print(f'\nGrid: N={N_fft}, omega_max={omega_max:.2f}, '
              f'Delta_omega={Delta_omega:.4f}, Delta_tau={Delta_tau}')


        # ── Residue-based IFT (exact, for comparison) ──
        if n_ext <= 1:
            spectrum_poles = find_spectrum_poles(propagator_data, num_params)
            print(f'\nSpectrum poles ({len(spectrum_poles)}):')
            for p in spectrum_poles:
                half = "upper" if p.imag > 0 else "lower"
                print(f'  {p:.6f}  ({half})')

            # Build tree-level S(omega) as fast_callable
            oe = SR.var('omega_e0')
            S_tree_sym = SR(0)
            for g in [g for g in kernel_groups if g['loop_number'] == 0]:
                ir = g['representative_ir']
                S_tree_sym += g['combined_prefactor'] * ir['integrand']
            S_tree_num = S_tree_sym.subs(num_params)
            f_tree_residue = fast_callable(S_tree_num.subs({oe: SR.var('omega')}),
                                           vars=[SR.var('omega')], domain=CDF)
            tau_residue = np.arange(float(-50), float(50) + Delta_tau/2, Delta_tau)
            # Detect DC offset: S(omega) -> constant as omega -> infinity
            # This constant contributes a delta(tau) term (shot noise).
            S_inf = complex(f_tree_residue(float(1e6))).real
            C_tree_residue = ift_via_residues(f_tree_residue, spectrum_poles, tau_residue).real
            # Add delta contribution at tau=0
            mid_r = len(tau_residue) // 2
            delta_weight = S_inf  # integral of constant = delta weight
            C_tree_residue[mid_r] += delta_weight / (tau_residue[1] - tau_residue[0])
            print(f'Tree (residue): C(0) = {C_tree_residue[mid_r]:.6e}')
            print(f'  (delta weight = {delta_weight:.6f}, smooth C(0) = {C_tree_residue[mid_r] - delta_weight/(tau_residue[1]-tau_residue[0]):.6e})')

        # ── Step 1: Tree-level (ell=0) — sum all tree spectra ──
        tree_groups = [g for g in kernel_groups if g['loop_number'] == 0]
        n_ext = k - 1  # number of independent external frequencies (stationary)

        if n_ext <= 1:
            # k<=2: evaluate on 1D grid directly
            Chat_tree = np.zeros(N_fft, dtype=complex)
            if tree_groups:
                n_tree_diags = sum(g['n_diagrams'] for g in tree_groups)
                for g in tree_groups:
                    Chat_tree += evaluate_kernel_group(g, num_params,
                                                      omega_grid=omega_grid)
                print(f'\n── Tree: {len(tree_groups)} unique kernel(s), '
                      f'{n_tree_diags} diagram(s)  ->  Chat(0) = {Chat_tree[mid]:.6e}')
        else:
            # k>=3: full nD spectrum evaluation + nD IFT
            # For k=3: evaluate S(w0, w1) on a 2D grid, then 2D IFT.
            grid_shape = tuple([N_fft] * n_ext)
            Chat_tree = np.zeros(grid_shape, dtype=complex)
            if tree_groups:
                n_tree_diags = sum(g['n_diagrams'] for g in tree_groups)
                print(f'\n\u2500\u2500 Tree: {len(tree_groups)} unique kernel(s), '
                      f'{n_tree_diags} diagram(s)')
                print(f'   Evaluating on {n_ext}D grid ({N_fft}^{n_ext} = {N_fft**n_ext} points)...')

                for g in tree_groups:
                    Chat_tree += evaluate_kernel_group(g, num_params,
                                                      omega_grid=omega_grid)

                mid_idx = tuple([mid] * n_ext)
                print(f'   Chat(0,...,0) = {Chat_tree[mid_idx]:.6e}')

        # ── Step 2: Loop — compute unique loop integrands, ──
        #    multiply by diagram-specific external factors, sum.
        if n_ext <= 1:
            Chat_loop = np.zeros(N_fft, dtype=complex)
        else:
            Chat_loop = np.zeros(grid_shape, dtype=complex)

        if loop_integrand_groups:
            n_unique = len(loop_integrand_groups)
            n_total_diags = sum(
                sum(g['n_diagrams'] for g in gs)
                for gs in loop_integrand_groups.values()
            )
            print(f'\n── Loop: {n_unique} unique loop integrand(s), '
                  f'{n_total_diags} diagram(s)')

            # Canonical variable names (matching prop_factors from canonicalize)
            sample_ir = next(iter(loop_integrand_groups.values()))[0]['representative_ir']
            n_ext = len(sample_ir['ext_freqs'])
            ext_vars_canonical = [SR.var(f'w_{ei}') for ei in range(n_ext)]
            ext_var = ext_vars_canonical[0] if ext_vars_canonical else None

            for li, (lsig, groups) in enumerate(loop_integrand_groups.items(), 1):
                ell = groups[0]['loop_number']
                total_diags = sum(g['n_diagrams'] for g in groups)

                # Build and evaluate the loop-only integrand once
                rep = groups[0]
                loop_vars_canonical = [SR.var(f'L_{lv}') for lv in range(ell)]
                loop_factors = [f for f in rep['prop_factors']
                                if _factor_depends_on(f, loop_vars_canonical)]

                loop_product = _build_factor_product(
                    loop_factors, propagator_data, omega, num_params)
                loop_var = loop_vars_canonical[0] if loop_vars_canonical else None

                loop_integral = _eval_loop_integral_on_grid(
                    loop_product, ext_var, loop_var,
                    omega_grid, num_params)

                # For each kernel group sharing this loop integrand,
                # multiply by its external factors and scalar prefactor
                Chat_this_integrand = np.zeros(N_fft, dtype=complex)
                for g in groups:
                    ext_factors = [f for f in g['prop_factors']
                                   if not _factor_depends_on(f, loop_vars_canonical)]
                    ext_product = _build_factor_product(
                        ext_factors, propagator_data, omega, num_params)
                    ext_on_grid = _eval_product_on_grid(ext_product, ext_var, omega_grid)

                    pf = float(g['combined_prefactor'].subs(num_params))
                    Chat_this_integrand += pf * ext_on_grid * loop_integral

                Chat_loop += Chat_this_integrand

                print(f'   Integrand {li}/{n_unique} (ell={ell}, '
                      f'{len(groups)} kernel group(s), '
                      f'{total_diags} diagrams): '
                      f'Chat(0) = {Chat_this_integrand[mid]:.6e}')

        # ── Step 3: Sum + IFT ──
        print(f'\n{"="*60}')
        print('Summary')
        print(f'{"="*60}')

        if n_ext <= 1:
            Chat_total = Chat_tree + Chat_loop
            tau_out, C_tree_tau = inverse_fourier(Chat_tree, omega_grid)
            _, C_total_tau = inverse_fourier(Chat_total, omega_grid)
            C_loop_tau = None
            if not np.all(Chat_loop == 0):
                _, C_loop_tau = inverse_fourier(Chat_loop, omega_grid)
            print(f'  Tree  Chat(0) = {Chat_tree[mid]:.6e}')
            print(f'  Loop  Chat(0) = {Chat_loop[mid]:.6e}')
            print(f'  Total Chat(0) = {Chat_total[mid]:.6e}')
        else:
            # k>=3: full nD IFT, then extract slices
            Chat_total = Chat_tree + Chat_loop
            tau_out, C_tree_tau_full = inverse_fourier(Chat_tree, omega_grid)
            _, C_total_tau_full = inverse_fourier(Chat_total, omega_grid)
            C_loop_tau = None

            # Extract 1D slices (fix other taus at 0 = mid index)
            C_tree_tau_slices = {}
            C_total_tau_slices = {}
            for s in range(n_ext):
                # Build index: all dimensions at mid except dimension s
                idx = [mid] * n_ext
                idx[s] = slice(None)
                idx = tuple(idx)
                C_tree_tau_slices[s] = C_tree_tau_full[idx]
                C_total_tau_slices[s] = C_total_tau_full[idx]

            mid_idx = tuple([mid] * n_ext)
            print(f'  Tree  Chat(0) = {Chat_tree[mid_idx]:.6e}')
            print(f'  Total Chat(0) = {Chat_total[mid_idx]:.6e}')

        # ── Plotting ──
        import matplotlib.pyplot as plt

        if n_ext <= 1:
            # k=2: 1D spectrum + 1D C(tau)
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            ax = axes[0]
            ax.plot(omega_grid, Chat_tree.real, 'b-', lw=1.5, label=r'Tree Re $\hat{C}$')
            if not np.all(Chat_loop == 0):
                ax.plot(omega_grid, Chat_loop.real, 'r-', lw=1.5, label=r'Loop Re $\hat{C}$')
            ax.set_xlabel(r'$\omega$', fontsize=13)
            ax.set_ylabel(r'$\hat{C}^{(' + str(k) + r')}(\omega)$', fontsize=13)
            ax.set_title(f'$k={k}$ Frequency-domain spectrum', fontsize=14)
            ax.legend(fontsize=11)
            ax.grid(True, alpha=0.3)
            ax.set_xlim(-2, 2)

            ax = axes[1]
            mask = np.abs(tau_out) <= 50
            ax.plot(tau_out[mask], C_tree_tau[mask].real, 'b-', lw=2, label='Tree (FFT)', alpha=0.6)
            if n_ext <= 1 and len(tau_residue) > 0:
                mask_r = np.abs(tau_residue) <= float(50)
                ax.plot(tau_residue[mask_r], C_tree_residue[mask_r], 'g--', lw=2, label='Tree (residue)')
            if C_loop_tau is not None:
                ax.plot(tau_out[mask], C_total_tau[mask].real, 'r--', lw=2, label='Tree + Loop')
            ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
            ax.set_ylabel(r'$C^{(' + str(k) + r')}(\tau)$', fontsize=13)
            ax.set_title(f'$k={k}$ Correlator', fontsize=14)
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
            ax.axhline(0, color='k', lw=0.5)

        else:
            # k>=3: slice plots
            fig, axes = plt.subplots(1, n_ext, figsize=(7 * n_ext, 5))
            if n_ext == 1:
                axes = [axes]

            mask = np.abs(tau_out) <= 50
            for s in range(n_ext):
                ax = axes[s]
                C_tree_s = C_tree_tau_slices[s]
                ax.plot(tau_out[mask], C_tree_s[mask].real, 'b-', lw=2, label='Tree')
                C_total_s = C_total_tau_slices[s]
                if not np.allclose(C_total_s, C_tree_s):
                    ax.plot(tau_out[mask], C_total_s[mask].real, 'r--', lw=2, label='Tree + Loop')

                other_labels = ', '.join(
                    rf'$\tau_{{{j+1}}}=0$' for j in range(n_ext) if j != s)
                ax.set_xlabel(rf'$\tau_{{{s+1}}}$', fontsize=13)
                ax.set_ylabel(r'$C^{(' + str(k) + r')}$', fontsize=13)
                ax.set_title(f'$k={k}$ slice: vary $\\tau_{{{s+1}}}$ ({other_labels})',
                             fontsize=13)
                ax.legend(fontsize=10)
                ax.grid(True, alpha=0.3)
                ax.axhline(0, color='k', lw=0.5)

        plt.tight_layout()
        plt.show()


### 8. Phase J — Time-domain hybrid pipeline (prototype)

Hybrid loop-kernel-reduction backend: uses frequency space only for unique
loop-kernel identification and algebraic grouping (reusing Phase I's
`group_diagrams_by_kernel`), and evaluates each diagram in the time domain
via vertex-time integration.

**MVP scope**: tree-level only.  Loop kernel reduction / caching /
contraction (Phases 3-5 of the full hybrid pipeline) are not yet
implemented — loop kernel groups are reported as 'skipped' and the caller
is expected to fall back to Phase I (cell 7.4) for those.

Expected outcome: Phase J tree contribution overlays the Phase I residue
curve exactly on the k=2 plot.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1  Evaluate tree-level contribution via Phase J
# ═══════════════════════════════════════════════════════════════
# Phase J uses explicit numerical quadrature (scipy.integrate.quad /
# nquad) on a fast_callable JIT'd integrand, with polytope bounds
# extracted from the retarded Heaviside factors.  It also enumerates
# the 2^|E| subsets of tree edges whose propagator entries have an
# instantaneous δ(t) component (the \tilde{n} × \delta n coupling in
# the MSR-JD Hawkes action): each subset's δ equalities pin
# integration variables; residual δ on external times comes back as
# a structured shot-noise spike that we insert into the τ grid below.
#
# For k=2 we evaluate on a single 1D τ grid with t_2 pinned to 0.
# For k=3 we evaluate on the two 1D slices that match the simulation
# cell's convention (cell 33):
#   slice 0: vary leaf-1\'s time, fix leaf-0 and leaf-2 at 0
#   slice 1: vary leaf-2\'s time, fix leaf-0 and leaf-1 at 0
# This matches ⟨field_a(0) · field_c(0) · field_b(τ)⟩ (slice 0) and
# ⟨field_a(0) · field_b(0) · field_c(τ)⟩ (slice 1) that the simulation
# extracts from the binned rate time series.
from msrjd.integration.time_domain import (
    compute_correction_td,
    integrate_tree_diagram,
    format_td_integral_latex,
    eval_delta_contributions_on_tau_grid,
)

if k == 1:
    print(f'Phase J: k=1 mean-field path not exercised — see cell 28 output.')
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None

elif k == 2:
    _origin_idx = 1  # pin t_2 → 0 so t_1 plays the role of tau
    _td_result = compute_correction_td(
        kernel_groups=kernel_groups,
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=_origin_idx,
    )

    n_tree_groups = sum(
        1 for g in _td_result['groups'] if g['handled_by'] == 'tree_evaluator'
    )
    n_skipped = len(_td_result['skipped_kernel_ids'])
    n_delta_total = len(_td_result['delta_contributions'])
    print(f'Phase J (k={k}): evaluated {n_tree_groups} tree kernel group(s), '
          f'skipped {n_skipped} loop kernel group(s), '
          f'{n_delta_total} shot-noise δ spike(s) produced.')
    for g in _td_result['groups']:
        extra = ''
        if 'n_delta_contributions' in g and g['n_delta_contributions']:
            extra = f"  δ-spikes={g['n_delta_contributions']}"
        print(f"   kernel ell={g['loop_number']}  "
              f"n_diagrams={g['n_diagrams']}  "
              f"handled_by={g['handled_by']}  "
              f"representation={g['representation']}  {g['reason']}{extra}")

    # Display the time-domain integral for each tree kernel group.
    _ext_syms = [SR.var(f't{j+1}_J') for j in range(k)]
    for _gi, _g in enumerate(_td_result['groups']):
        if _g['handled_by'] != 'tree_evaluator':
            continue
        _td_diag = _g_idx = None
        for _kg in kernel_groups:
            if _kg.get('signature') == _g.get('kernel_id'):
                _td_diag = _kg['diagrams'][0]
                _g_idx = _kg
                break
        if _td_diag is None:
            continue
        _per_group_result = integrate_tree_diagram(
            typed_diagram=_td_diag,
            representative_ir=_g_idx['representative_ir'],
            propagator_data=propagator_data,
            combined_prefactor=_g_idx['combined_prefactor'],
            ext_time_vars=_ext_syms,
            num_params=num_params,
            origin_leaf_idx=_origin_idx,
        )
        _label = f'Phase J group {_gi}  (ell={_g["loop_number"]}, n={_g["n_diagrams"]})'
        _latex_str = format_td_integral_latex(
            _per_group_result,
            typed_diagram=_td_diag,
            combined_prefactor=_g_idx['combined_prefactor'],
            ext_time_vars=_ext_syms,
            label=_label,
        )
        display(Math(_latex_str))

    # Evaluate on a single 1D τ grid (t_2 pinned)
    # (Numerical τ-grid evaluation is in cell 8.1c below.)

elif k == 3:
    # k=3: no pinning — Phase J returns C(t_1, t_2, t_3) and we evaluate
    # slices matching the simulation conventions.
    #
    # For each value in TAU_FIXED_LIST we evaluate TWO slices:
    #   slice s=0:  vary leaf 1 time → C(0, τ, τ_fixed)
    #   slice s=1:  vary leaf 2 time → C(0, τ_fixed, τ)
    # τ_fixed = 0 recovers the two canonical "other τ pinned to origin"
    # slices; nonzero τ_fixed lets us peek at off-diagonal cuts of the
    # 2D cumulant surface.
    TAU_FIXED_LIST = [0.0, 5.0]   # can be extended / edited by the user

    _td_result = compute_correction_td(
        kernel_groups=kernel_groups,
        propagator_data=propagator_data,
        k=k,
        num_params=num_params,
        origin_leaf_idx=None,
    )

    n_tree_groups = sum(
        1 for g in _td_result['groups'] if g['handled_by'] == 'tree_evaluator'
    )
    n_skipped = len(_td_result['skipped_kernel_ids'])
    n_delta_total = len(_td_result['delta_contributions'])
    print(f'Phase J (k={k}): evaluated {n_tree_groups} tree kernel group(s), '
          f'skipped {n_skipped} loop kernel group(s), '
          f'{n_delta_total} shot-noise δ spike(s) produced.')
    for g in _td_result['groups']:
        extra = ''
        if 'n_delta_contributions' in g and g['n_delta_contributions']:
            extra = f"  δ-spikes={g['n_delta_contributions']}"
        print(f"   kernel ell={g['loop_number']}  "
              f"n_diagrams={g['n_diagrams']}  "
              f"handled_by={g['handled_by']}  "
              f"representation={g['representation']}  {g['reason']}{extra}")

    # Display the time-domain integral for each tree kernel group.
    _ext_syms = [SR.var(f't{j+1}_J') for j in range(k)]
    for _gi, _g in enumerate(_td_result['groups']):
        if _g['handled_by'] != 'tree_evaluator':
            continue
        _td_diag = _g_idx = None
        for _kg in kernel_groups:
            if _kg.get('signature') == _g.get('kernel_id'):
                _td_diag = _kg['diagrams'][0]
                _g_idx = _kg
                break
        if _td_diag is None:
            continue
        _per_group_result = integrate_tree_diagram(
            typed_diagram=_td_diag,
            representative_ir=_g_idx['representative_ir'],
            propagator_data=propagator_data,
            combined_prefactor=_g_idx['combined_prefactor'],
            ext_time_vars=_ext_syms,
            num_params=num_params,
            origin_leaf_idx=None,
        )
        _label = f'Phase J group {_gi}  (ell={_g["loop_number"]}, n={_g["n_diagrams"]})'
        _latex_str = format_td_integral_latex(
            _per_group_result,
            typed_diagram=_td_diag,
            combined_prefactor=_g_idx['combined_prefactor'],
            ext_time_vars=_ext_syms,
            label=_label,
        )
        display(Math(_latex_str))

    # Phase J callable signature is (t_1, t_2, t_3).
    # For each (s, τ_fixed) combo:
    #   s = 0:  vary leaf 1 → C(0, τ, τ_fixed)   (τ_other = t_3 fixed)
    #   s = 1:  vary leaf 2 → C(0, τ_fixed, τ)   (τ_other = t_2 fixed)
    # τ_fixed = 0 gives the canonical slices comparable with the
    # simulation's `τ_other = 0` plots; τ_fixed = +5 probes an
    # off-diagonal cut through the 3-point surface.
    # (Numerical τ-grid evaluation is in cell 8.1c below.)
else:
    print(f'Phase J prototype currently supports k in (2, 3); '
          f'current k={k}.  Nothing to do.')
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1b  Symbolic decomposition: δ vs smooth propagator factors
# ═══════════════════════════════════════════════════════════════
# Before numerical integration, show how each tree diagram's integrand
# decomposes when we split each retarded propagator into its δ(t) and
# smooth parts:
#
#   G_R[pi, ri](Δt) = c_δ · δ(Δt) + Θ(Δt) · G_smooth(Δt)
#
# Distributing the product over |E| edges gives 2^|E| terms. For each
# term, δ factors collapse integration variables, and the remaining
# smooth factors define the integrand for numerical quadrature.

from msrjd.integration.time_domain import build_G_t_matrix, G_t_delta_coeff
from msrjd.integration.time_domain.final_integral import _lookup_prop_indices

if k >= 2:
    _t_sym = SR.var('_t_decomp_')
    _G_obj = build_G_t_matrix(propagator_data, _t_sym, num_params=num_params)
    _G_smooth = _G_obj['smooth']
    _G_delta = _G_obj['delta']
    _resp_names = [str(g) for g in ft.ring().gens()][:ft._n_tilde]
    _phys_names = [str(g) for g in ft.ring().gens()][ft._n_tilde:]

    print('Propagator decomposition: G_R[pi, ri](t) = c_δ · δ(t) + Θ(t) · G_smooth(t)')
    print('─' * 70)
    for pi in range(propagator_data['nf']):
        for ri in range(propagator_data['nf']):
            c_d = G_t_delta_coeff(_G_obj, pi, ri)
            smooth_entry = _G_smooth[pi, ri]
            if smooth_entry == 0 and abs(complex(c_d)) < 1e-15:
                continue
            c_d_str = f'{complex(c_d).real:+.4f}' if abs(complex(c_d)) > 1e-15 else '  0   '
            label = f'G[{_phys_names[pi]:>4}, {_resp_names[ri]:>4}]'
            print(f'  {label}:  c_δ = {c_d_str}   smooth(t) = {smooth_entry}')
    print()

    _ext_syms = [SR.var(f't{j+1}') for j in range(k)]
    _origin_idx = 1 if k == 2 else None

    for gi, g in enumerate(kernel_groups):
        if g['loop_number'] != 0:
            continue
        td = g['diagrams'][0]
        D = td.prediagram[0]
        leaves = list(td.prediagram[2])
        leaf_set = set(leaves)
        internal = [v for v in D.vertices() if v not in leaf_set]
        edges = list(D.edges())
        cp_num = float(g['combined_prefactor'].subs(num_params))

        vertex_time = {}
        for j, lf in enumerate(leaves):
            t_ext = _ext_syms[j]
            if _origin_idx is not None and j == _origin_idx:
                t_ext = SR(0)
            vertex_time[lf] = t_ext
        int_vars = []
        for v in internal:
            s_v = SR.var(f's_{v}')
            vertex_time[v] = s_v
            int_vars.append(s_v)

        print(f'{"═" * 70}')
        print(f'Kernel group {gi}  (n={g["n_diagrams"]}, pf = {cp_num:+.6f})')
        print(f'{"═" * 70}')
        print(f'  Vertex times: ' + ', '.join(
            f'v{v}={vertex_time[v]}' for v in sorted(vertex_time)))
        print(f'  Integration vars: {[str(s) for s in int_vars]}')
        print(f'  Edges:')

        edge_info = []
        for (u, v, lbl) in edges:
            ri, pi = _lookup_prop_indices(td, (u, v, lbl))
            dt = vertex_time[v] - vertex_time[u]
            c_d = G_t_delta_coeff(_G_obj, pi, ri)
            _t_var = _G_obj.get('t_var', _t_sym)
            smooth_expr = SR(_G_smooth[pi, ri]).subs({_t_var: dt})
            has_delta = abs(complex(c_d)) > 1e-15
            edge_info.append({'u': u, 'v': v, 'ri': ri, 'pi': pi,
                              'dt': dt, 'c_delta': c_d, 'smooth': smooth_expr,
                              'has_delta': has_delta})
            ds = f'c_δ={complex(c_d).real:+.4f}' if has_delta else 'c_δ=0'
            print(f'    ({u}→{v}): G[{_phys_names[pi]},{_resp_names[ri]}]'
                  f'(Δt={dt})  {ds}')

        n_edges = len(edge_info)
        n_with_delta = sum(1 for e in edge_info if e['has_delta'])
        print(f'\n  2^{n_with_delta} = {2**n_with_delta} nonzero subsets '
              f'(from {n_with_delta} edges with δ):')
        print()

        for subset_bits in range(2**n_edges):
            delta_edges = [i for i in range(n_edges) if (subset_bits >> i) & 1]
            smooth_edges = [i for i in range(n_edges) if not ((subset_bits >> i) & 1)]
            if any(not edge_info[i]['has_delta'] for i in delta_edges):
                continue

            delta_coeff = SR(cp_num)
            delta_strs = []
            for i in delta_edges:
                delta_coeff *= SR(complex(edge_info[i]['c_delta']).real)
                delta_strs.append(f'δ({edge_info[i]["dt"]})')

            # Solve δ constraints
            subs = {}
            residuals = []
            remaining = list(int_vars)
            for i in delta_edges:
                dt_sub = SR(edge_info[i]['dt']).subs(subs)
                solved = False
                for iv in remaining:
                    if iv in dt_sub.variables():
                        sol = solve(dt_sub == 0, iv, solution_dict=True)
                        if sol:
                            subs[iv] = sol[0][iv]
                            remaining.remove(iv)
                            solved = True
                            break
                if not solved:
                    residuals.append(str(dt_sub))

            # Classify smooth factors: inner (depends on remaining int vars) vs outer
            remaining_set = set(remaining)
            inner = []
            outer = []
            for i in smooth_edges:
                e = edge_info[i]
                dt_sub = SR(e['dt']).subs(subs)
                deps = set(dt_sub.variables()) & remaining_set
                tag = f'G_sm[{_phys_names[e["pi"]]},{_resp_names[e["ri"]]}]({dt_sub})'
                theta = f'Θ({dt_sub})'
                if deps:
                    inner.append((tag, theta))
                else:
                    outer.append((tag, theta))

            label = 'S=∅' if not delta_edges else f'S={{{",".join(str(i) for i in delta_edges)}}}'
            print(f'  ── {label} ──')
            if delta_strs:
                print(f'     δ:  {" × ".join(delta_strs)}')
            if subs:
                print(f'     →  {", ".join(f"{k_} = {v_}" for k_, v_ in subs.items())}')
            if residuals:
                res_str = ', '.join(residuals)
                if all(r == '0' for r in residuals):
                    print(f'     →  residual: {res_str} = 0 (trivial → degenerate δ on slice)')
                else:
                    print(f'     →  residual ext-time equality: {res_str} = 0')

            coeff_val = float(SR(delta_coeff).real())
            if remaining:
                int_str = ' '.join(f'∫d{s}' for s in remaining)
                if outer:
                    outer_str = ' × '.join(f[0] for f in outer)
                    outer_theta = ' × '.join(f[1] for f in outer)
                    inner_str = ' × '.join(f'{f[0]} × {f[1]}' for f in inner)
                    print(f'     = {coeff_val:+.6f}')
                    print(f'       × [{outer_str}] × [{outer_theta}]')
                    print(f'       × {int_str} [{inner_str}]')
                else:
                    inner_str = ' × '.join(f'{f[0]} × {f[1]}' for f in inner)
                    print(f'     = {coeff_val:+.6f} × {int_str} [{inner_str}]')
            else:
                if outer:
                    outer_str = ' × '.join(f'{f[0]} × {f[1]}' for f in outer)
                    print(f'     = {coeff_val:+.6f} × {outer_str}')
                else:
                    print(f'     = {coeff_val:+.6f}  (constant)')
            print()

    print('─' * 70)
    print('Inspect the above, then run the next cell for numerical evaluation.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.1c  Numerical evaluation of Phase J on the τ grid
# ═══════════════════════════════════════════════════════════════
# This cell evaluates the Phase J callable (built in cell 8.1) at
# each point on the τ grid.  THIS is the slow step — each point
# invokes scipy.integrate.quad on the stripped integrand.
#
# The symbolic decomposition (cell 8.1b) should be inspected first
# to verify the integrand structure looks correct.
from msrjd.integration.time_domain import eval_delta_contributions_on_tau_grid

if k == 1:
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None

elif k == 2:
    _total_C_fn = _td_result['total_C']
    tau_phase_j = np.array(tau_residue, dtype=float)
    C_tree_phase_j = np.zeros_like(tau_phase_j, dtype=float)
    for _i, _tv in enumerate(tau_phase_j):
        _val = _total_C_fn(float(_tv), 0.0)
        C_tree_phase_j[_i] = float(np.real(_val))
    _delta_array = eval_delta_contributions_on_tau_grid(
        _td_result['delta_contributions'], tau_phase_j, free_ext_dim=1,
    )
    C_tree_phase_j = C_tree_phase_j + _delta_array.real
    C_tree_phase_j_slices = None
    _mid_j = len(tau_phase_j) // 2
    print(f"Phase J tree  C(0) = {C_tree_phase_j[_mid_j]:.6e}")
    if 'C_tree_residue' in globals() and C_tree_residue is not None and len(C_tree_residue) == len(tau_phase_j):
        _abs_err = float(np.max(np.abs(C_tree_phase_j - C_tree_residue)))
        print(f"max |Phase J - Phase I residue| = {_abs_err:.3e}")

elif k == 3:
    _total_C_fn = _td_result['total_C']
    tau_phase_j = np.array(tau_residue, dtype=float)
    C_tree_phase_j_slices = {}
    for _tau_fixed in TAU_FIXED_LIST:
        for _s in (0, 1):
            _C_slice = np.zeros_like(tau_phase_j, dtype=float)
            for _i, _tv in enumerate(tau_phase_j):
                if _s == 0:
                    _val = _total_C_fn(0.0, float(_tv), float(_tau_fixed))
                else:
                    _val = _total_C_fn(0.0, float(_tau_fixed), float(_tv))
                _C_slice[_i] = float(np.real(_val))
            _vary_index = 1 if _s == 0 else 2
            _fixed_other_index = 2 if _s == 0 else 1
            _fixed_dict = {0: 0.0, _fixed_other_index: float(_tau_fixed)}
            _delta_array = eval_delta_contributions_on_tau_grid(
                _td_result['delta_contributions'], tau_phase_j,
                free_ext_dim=3, vary_index=_vary_index,
                fixed_values=_fixed_dict,
            )
            _C_slice = _C_slice + _delta_array.real
            C_tree_phase_j_slices[(_s, float(_tau_fixed))] = _C_slice

    C_tree_phase_j = C_tree_phase_j_slices[(0, 0.0)]
    _mid_j = len(tau_phase_j) // 2
    print()
    for _tau_fixed in TAU_FIXED_LIST:
        print(f"Phase J tree  τ_fixed={_tau_fixed:+g}  "
              f"slice 0 at τ=0 = "
              f"{C_tree_phase_j_slices[(0, float(_tau_fixed))][_mid_j]:.6e}")
        print(f"Phase J tree  τ_fixed={_tau_fixed:+g}  "
              f"slice 1 at τ=0 = "
              f"{C_tree_phase_j_slices[(1, float(_tau_fixed))][_mid_j]:.6e}")

else:
    tau_phase_j = None
    C_tree_phase_j = None
    C_tree_phase_j_slices = None


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.2  Overlay Phase J tree result on the k-point comparison plot
# ═══════════════════════════════════════════════════════════════
# For k=2 this overlays Phase J on top of Phase I\'s FFT IFT and
# residue IFT curves.
# For k=3 we plot Phase J on its own (Phase I was skipped in cell 28)
# as two slice directions × one panel per τ_fixed value. The slice
# dictionary C_tree_phase_j_slices is keyed by (slice_idx, τ_fixed)
# from cell 30.
import matplotlib.pyplot as plt

if k == 2 and C_tree_phase_j is not None:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    if C_tree_tau is not None:
        _mask_fft = np.abs(tau_out) <= 50
        ax.plot(tau_out[_mask_fft], C_tree_tau[_mask_fft].real,
                'b-', lw=2, alpha=0.5, label='Phase I tree (FFT IFT)')

    if 'tau_residue' in globals() and C_tree_residue is not None and len(tau_residue) > 0:
        _mask_r = np.abs(tau_residue) <= 50
        ax.plot(tau_residue[_mask_r], C_tree_residue[_mask_r],
                'g--', lw=2, label='Phase I tree (residue IFT)')

    _mask_j = np.abs(tau_phase_j) <= 50
    ax.plot(tau_phase_j[_mask_j], C_tree_phase_j[_mask_j],
            'm:', lw=2.5, label='Phase J tree (time domain)')

    ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
    ax.set_ylabel(r'$C^{(2)}(\tau)$', fontsize=13)
    ax.set_title('Tree-level 2-point correlator — Phase I vs Phase J',
                 fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    plt.tight_layout()
    plt.show()

elif k == 3 and C_tree_phase_j_slices is not None:
    # Group slices by slice index (0 = vary τ₁, 1 = vary τ₂)
    # and plot one subplot per (slice, τ_fixed) pair.
    _tau_fixed_values = sorted({k_[1] for k_ in C_tree_phase_j_slices.keys()})
    _n_fixed = len(_tau_fixed_values)
    fig, axes = plt.subplots(_n_fixed, 2, figsize=(14, 5 * _n_fixed),
                             squeeze=False)
    _mask_j = np.abs(tau_phase_j) <= 50

    _colors = ['m', 'darkorange', 'teal', 'purple']
    for _row, _tau_fixed in enumerate(_tau_fixed_values):
        for _s in (0, 1):
            ax = axes[_row, _s]
            _curve = C_tree_phase_j_slices.get((_s, float(_tau_fixed)))
            if _curve is None:
                continue
            _color = _colors[_row % len(_colors)]
            ax.plot(tau_phase_j[_mask_j], _curve[_mask_j],
                    color=_color, lw=2,
                    label=f'Phase J tree (τ_fixed={_tau_fixed:+g})')
            _vary_idx = _s + 1
            _fixed_idx = 2 - _s
            _xlabel = rf'$\tau_{{{_vary_idx}}}$'
            if _s == 0:
                _title = rf'vary $\tau_1$,  $\tau_2={_tau_fixed:+g}$'
            else:
                _title = rf'vary $\tau_2$,  $\tau_1={_tau_fixed:+g}$'
            ax.set_xlabel(_xlabel, fontsize=13)
            ax.set_ylabel(r'$C^{(3)}$', fontsize=13)
            ax.set_title(_title, fontsize=12)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.axhline(0, color='k', lw=0.5)
    fig.suptitle(
        'Tree-level 3-point correlator — Phase J time-domain slices',
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

else:
    print(f'Phase J overlay plot skipped (k={k}).')


### 9. Simulation comparison

Simulate the 2-population nonlinear Hawkes process with the same
fundamental parameters and compare:
- Mean firing rates: simulation vs MF + 1-loop
- Cross-covariance $C_{12}(\tau)$: simulation vs tree + 1-loop

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9.  Simulation of the 2-population nonlinear Hawkes process
# ═══════════════════════════════════════════════════════════════
import numpy as np
from numpy.random import default_rng

# ── Simulation parameters ──
# For k >= 3 the 3-point cumulant estimator needs MUCH more data than
# the 2-point. The Euler timestep dt_sim can be safely coarsened
# (dt_sim / tau < 0.05 is fine for linear ODE stability/accuracy),
# and the bin width dt_bin can be increased (the signal's timescale
# is ~tau, so dt_bin = 1 still resolves it) for lower per-bin noise.
if k <= 2:
    T_sim    = float(200000)    # 200k — adequate for k=2
    dt_sim   = float(0.02)     # fine Euler step
    dt_bin   = float(0.25)     # fine time resolution
else:
    T_sim    = float(50_000_000)  # 50M — needed for k=3 SNR ~ 3
    dt_sim   = float(0.5)     # coarse but accurate (dt/tau = 5%)
    dt_bin   = float(1.0)     # coarser bins → 4× less shot noise
tau_max  = float(50)       # max lag for covariance
rng      = default_rng(int(42))

# ── Model parameters (from fundamental dict) ──
E_sim   = [float(x) for x in fundamental['E']]
w_sim   = [[float(x) for x in row] for row in fundamental['w']]
tau_sim = float(fundamental['tau'])
# No 'a' parameter for linear phi
npop_sim = len(E_sim)


def phi_sim(v_val):
    """Transfer function: phi(v) = v (linear)."""
    return v_val

import time as _time
# Import the Numba-JIT compiled simulator from a plain .py file.
# (It MUST live in a .py file, not a notebook cell, because SageMath's
# preparser converts integer literals like 0 to Integer(0) which
# Numba's type inference cannot handle.)
from models.hawkes_sim_numba import sim_hawkes_numba


# ═══════════════════════════════════════════════════════════════
# Euler-step simulation with Poisson spike draws (Numba-JIT)
# ═══════════════════════════════════════════════════════════════
n_steps = int(T_sim / dt_sim)
# Fix: use round() so dt_bin / dt_sim near-integers snap correctly,
# then compute the ACTUAL bin width dt_bin_eff = bin_size_steps × dt_sim.
# All downstream analysis uses dt_bin_eff, not the nominal dt_bin.
bin_size_steps = max(int(round(dt_bin / dt_sim)), 1)
dt_bin_eff = bin_size_steps * dt_sim
n_bins = n_steps // bin_size_steps
v_init = np.array([float(vstar_vals[i]) for i in range(npop_sim)])
E_arr = np.array(E_sim, dtype=float)
W_arr = np.array(w_sim, dtype=float)

print(f'Simulating T={T_sim:.0f}, dt_sim={dt_sim}, dt_bin(nominal)={dt_bin}, '
      f'n_steps={n_steps:,}, n_bins={n_bins:,} ...')

# JIT warmup (first call compiles — takes ~2s).
# All args must be plain Python int / float (not Sage Integer).
sim_hawkes_numba(int(1000), float(dt_sim), float(tau_sim), E_arr, W_arr,
                 v_init.copy(), int(bin_size_steps), int(100), int(0))

_t0_sim = _time.perf_counter()
binned_counts, total_spikes = sim_hawkes_numba(
    int(n_steps), float(dt_sim), float(tau_sim), E_arr, W_arr,
    v_init.copy(), int(bin_size_steps), int(n_bins), int(42),
)
_elapsed_sim = _time.perf_counter() - _t0_sim

# T_used excludes any discarded tail from non-divisible n_steps.
T_used = n_bins * dt_bin_eff
sim_rates = total_spikes / T_used
print(f'Done in {_elapsed_sim:.1f}s ({n_steps/_elapsed_sim:,.0f} steps/s).')
if abs(dt_bin_eff - dt_bin) > 1e-12:
    print(f'  NOTE: effective bin width dt_bin_eff = {dt_bin_eff} '
          f'(nominal dt_bin = {dt_bin}, ratio dt_bin/dt_sim = {dt_bin/dt_sim:.4f})')
for i in range(npop_sim):
    print(f'  Pop {i+1}: {int(total_spikes[i])} spikes, rate = {sim_rates[i]:.4f}')


# (The old binned_rates = binned_counts / dt_bin code has been removed.
#  The cumulant estimator now works directly on binned_counts with
#  proper factorial corrections — see models/cumulant_estimator.py.)

# ═══════════════════════════════════════════════════════════════
# Compute k-point cumulant from simulation to match theory
# ═══════════════════════════════════════════════════════════════
max_lag_bins = int(tau_max / dt_bin_eff)
n_fft_sim = 2 * n_bins
tau_sim_grid = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin_eff
n_ext = k - 1  # number of independent time differences (stationary)

# Population indices for external fields (0-based)
pop_indices = [ef[1] - 1 for ef in external_fields]
field_labels = [f'{ef[0]}_{ef[1]}' for ef in external_fields]
corr_label = ', '.join(field_labels)

def extract_lag_window(xcorr_full, max_lag_bins, n_fft_total):
    out = np.zeros(2 * max_lag_bins + 1)
    for idx_lag, lag in enumerate(range(-max_lag_bins, max_lag_bins + 1)):
        out[idx_lag] = xcorr_full[lag % n_fft_total]
    return out

if k == 1:
    print(f'k=1: mean rates comparison only')
    C_sim_slices = {}

elif k >= 2:
    # ── General k-point cumulant estimator ──────────────────────────
    # Uses factorial-moment correction for same-population fields at
    # the same time bin: replaces n^m with n(n-1)...(n-m+1), which
    # removes the self-spike shot-noise contribution that would
    # otherwise make the estimator dt_bin-dependent. See
    # models/cumulant_estimator.py for the algorithm.
    from models.cumulant_estimator import compute_kpoint_slice
    from models.cumulant_estimator import compute_kpoint_slice_direct

    # Toggle: True = direct loop (slow but transparent), False = FFT (fast)
    USE_DIRECT = False
    _slice_fn = compute_kpoint_slice_direct if USE_DIRECT else compute_kpoint_slice
    print(f'  Estimator: {"direct (non-FFT)" if USE_DIRECT else "FFT-based"}')

    if 'TAU_FIXED_LIST' not in globals():
        TAU_FIXED_LIST = [0.0] if k == 2 else [0.0, 5.0]

    C_sim_slices = {}
    for tau_fixed in TAU_FIXED_LIST:
        # Snap to the actual bin grid (tau_fixed may not be an exact
        # multiple of dt_bin_eff; use the nearest bin index).
        tau_fixed_bins = int(round(tau_fixed / dt_bin_eff))
        tau_fixed_actual = tau_fixed_bins * dt_bin_eff
        for s in range(n_ext):
            # Build the lag_bins list for this slice.
            # Convention: field a (= external_fields[0]) is the reference
            # (lag = 0). The sweep axis gets lag = None.
            if k == 2:
                lag_bins = [0, None]  # a fixed, b sweeps
            elif k == 3:
                if s == 0:
                    # vary tau1 (= lag of field b from a), fix tau2 = tau_fixed
                    lag_bins = [0, None, tau_fixed_bins]
                else:
                    # vary tau2 (= lag of field c from a), fix tau1 = tau_fixed
                    lag_bins = [0, tau_fixed_bins, None]
            else:
                raise NotImplementedError(f'k={k} slice layout not yet configured')

            tau_sim_grid, C_slice = _slice_fn(
                binned_counts, float(dt_bin_eff),
                [int(p) for p in pop_indices],
                lag_bins, int(max_lag_bins),
            )
            if k == 2:
                C_sim_slices[s] = C_slice
            else:
                C_sim_slices[(s, float(tau_fixed))] = C_slice

    print(f'Cumulant slices computed (k={k}, '
          f'{len(C_sim_slices)} slice(s), factorial-corrected).')


print(f'\nSimulated rates: {[f"{r:.4f}" for r in sim_rates]}')
print(f'MF rates:        {[f"{nstar_vals[i]:.4f}" for i in range(npop_sim)]}')


# ═══════════════════════════════════════════════════════════════
# Comparison plots
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

# For k=1, k=2: single slice comparison + rates panel.
# For k=3: one row per τ_fixed value (so the user can see how both
# slice directions compare against Phase J at several fixed values).

mf_rates = [float(nstar_vals[i]) for i in range(npop_sim)]

def _plot_mean_rates(ax):
    x_pos = np.arange(npop_sim)
    width = 0.25
    ax.bar(x_pos - width, sim_rates, width, label='Simulation',
           color='#4CAF50', edgecolor='k', linewidth=0.5)
    ax.bar(x_pos, mf_rates, width, label='Mean-field',
           color='#2196F3', edgecolor='k', linewidth=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
    ax.set_ylabel('Firing rate', fontsize=12)
    ax.set_title('Mean firing rates', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')


if n_ext <= 1:
    # k=1 or k=2: single row with rates + one cumulant panel
    n_plots = max(1, n_ext)
    fig, axes = plt.subplots(1, 1 + n_plots,
                             figsize=(7 * (1 + n_plots), 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    _plot_mean_rates(axes[0])

    for s in range(n_ext):
        ax = axes[1 + s]
        # Old schema (k=2): C_sim_slices keyed by s alone
        _sim = C_sim_slices.get(s)
        if _sim is not None:
            ax.plot(tau_sim_grid, _sim, 'k-', lw=1.5, alpha=0.7,
                    label='Simulation')

        mask_an = np.abs(tau_out) <= tau_max
        if C_tree_tau is not None:
            ax.plot(tau_out[mask_an], C_tree_tau[mask_an].real,
                    'b-', lw=2, label='Phase I tree')
        if C_loop_tau is not None:
            ax.plot(tau_out[mask_an], C_total_tau[mask_an].real,
                    'r--', lw=2, label='Phase I tree + 1-loop')

        if tau_phase_j is not None and C_tree_phase_j is not None:
            _mask_j = np.abs(tau_phase_j) <= tau_max
            ax.plot(tau_phase_j[_mask_j], C_tree_phase_j[_mask_j],
                    'm:', lw=2.5, label='Phase J tree (time domain)')

        ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
        is_auto = (len(external_fields) >= 2
                   and external_fields[0] == external_fields[1])
        ctype = 'Auto' if is_auto else 'Cross'
        ax.set_title(f'{ctype}-covariance: {corr_label}', fontsize=13)
        ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', lw=0.5)

    plt.tight_layout()
    plt.show()

else:
    # k>=3: one row per τ_fixed value. Row 0 also carries the mean
    # rates subplot in the leftmost column; later rows leave that slot
    # blank.
    _tau_fixed_values = sorted({k_[1] for k_ in C_sim_slices.keys()}) \
        if C_sim_slices else [0.0]
    _n_rows = len(_tau_fixed_values)
    fig, axes = plt.subplots(_n_rows, 1 + n_ext,
                             figsize=(7 * (1 + n_ext), 5 * _n_rows),
                             squeeze=False)
    _plot_mean_rates(axes[0, 0])
    # Leave any remaining leftmost slots blank
    for _r in range(1, _n_rows):
        axes[_r, 0].axis('off')

    for _r, _tau_fixed in enumerate(_tau_fixed_values):
        for s in range(n_ext):
            ax = axes[_r, 1 + s]

            # Simulation
            _sim = C_sim_slices.get((s, float(_tau_fixed)))
            if _sim is not None:
                ax.plot(tau_sim_grid, _sim, 'k-', lw=1.5, alpha=0.7,
                        label='Simulation')

            # Phase J theory (time-domain slices)
            _pj_key = (s, float(_tau_fixed))
            if ('C_tree_phase_j_slices' in globals()
                    and C_tree_phase_j_slices is not None
                    and _pj_key in C_tree_phase_j_slices
                    and tau_phase_j is not None):
                _mask_j = np.abs(tau_phase_j) <= tau_max
                ax.plot(tau_phase_j[_mask_j],
                        C_tree_phase_j_slices[_pj_key][_mask_j],
                        'm-', lw=2, label='Phase J tree (time domain)')

            # Phase I is skipped for k>=3 (see cell 28).

            other_label = (rf'$\tau_{{{2-s}}}={_tau_fixed:+g}$')
            ax.set_xlabel(rf'$\tau_{{{s+1}}}$', fontsize=13)
            ax.set_title(f'$k={k}$ slice: vary $\\tau_{{{s+1}}}$, '
                         f'{other_label}', fontsize=12)
            ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.axhline(0, color='k', lw=0.5)

    plt.tight_layout()
    plt.show()

# ── Summary ──
print(f'\n{"="*60}')
print('Comparison summary')
print(f'{"="*60}')
print(f'  k = {k}, external_fields = {external_fields}')
for i in range(npop_sim):
    mf = nstar_vals[i]
    sim_r = sim_rates[i]
    pct = 100 * (sim_r - mf) / mf if mf > 0 else 0
    print(f'  Pop {i+1}:  sim={sim_r:.4f}  MF={mf:.4f}  diff={pct:+.2f}%')
print(f'\n  T={T_sim:.0f}, dt={dt_sim}, bins={n_bins}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.3  Heatmap comparison: theory vs simulation on the (τ₁, τ₂) grid
# ═══════════════════════════════════════════════════════════════
# Both theory and sim heatmaps are built row-by-row using the SAME
# convention: fix τ₁ (= lag of field b from field a), sweep τ₂
# (= lag of field c from field a). This ensures identical lag
# conventions and normalization between theory and sim.

import matplotlib.pyplot as plt

if k == 3 and '_td_result' in dir():
    # ── Settings ──
    HEATMAP_TAU_MAX  = 30.0
    HEATMAP_TAU_STEP = 1.0

    tau_1d = np.arange(-HEATMAP_TAU_MAX, HEATMAP_TAU_MAX + HEATMAP_TAU_STEP/2,
                       HEATMAP_TAU_STEP)
    n_tau = len(tau_1d)

    # ── Theory heatmap: row-by-row, same convention as sim ──
    # For each τ₁ row, evaluate total_C(t_a=0, t_b=τ₁, t_c=τ₂) for
    # each τ₂ in tau_1d.
    _total_C_fn = _td_result['total_C']
    C_theory_2d = np.zeros((n_tau, n_tau))
    print(f'Evaluating theory heatmap ({n_tau}x{n_tau} = {n_tau**2} points)...')
    for i, t1 in enumerate(tau_1d):
        for j, t2 in enumerate(tau_1d):
            C_theory_2d[i, j] = float(np.real(
                _total_C_fn(0.0, float(t1), float(t2))
            ))
    print(f'  Done. Range: [{C_theory_2d.min():.4e}, {C_theory_2d.max():.4e}]')

    # ── Sim heatmap: row-by-row using compute_kpoint_slice ──
    # For each τ₁ row, call compute_kpoint_slice with:
    #   lag_bins = [0, t1_bins, None]   (field a at 0, b at τ₁, c sweeps)
    from models.cumulant_estimator import compute_kpoint_slice
    heatmap_lag_bins = int(HEATMAP_TAU_MAX / dt_bin_eff)

    C_sim_2d = np.zeros((n_tau, n_tau))
    print(f'Evaluating sim heatmap ({n_tau} rows)...')
    for i, t1_val in enumerate(tau_1d):
        t1_bins = int(round(t1_val / dt_bin_eff))
        lag_bins_row = [0, t1_bins, None]
        try:
            tau_row, c_row = compute_kpoint_slice(
                binned_counts, float(dt_bin_eff),
                [int(p) for p in pop_indices],
                lag_bins_row, heatmap_lag_bins,
            )
            C_sim_2d[i, :] = np.interp(tau_1d, tau_row, c_row)
        except Exception as e:
            print(f'  row {i} (tau1={t1_val}): {e}')
            C_sim_2d[i, :] = 0.0
    print(f'  Done. Range: [{C_sim_2d.min():.4e}, {C_sim_2d.max():.4e}]')

    # ── 3-panel plot ──
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    extent = [tau_1d[0], tau_1d[-1], tau_1d[0], tau_1d[-1]]

    # Panel 1: Theory on its own scale
    th_max = max(abs(C_theory_2d.min()), abs(C_theory_2d.max()), 1e-15)
    im0 = axes[0].imshow(
        C_theory_2d.T, origin='lower', aspect='equal', extent=extent,
        cmap='RdBu_r', vmin=-th_max, vmax=th_max,
    )
    axes[0].set_xlabel(r'$\tau_1$', fontsize=13)
    axes[0].set_ylabel(r'$\tau_2$', fontsize=13)
    axes[0].set_title('Theory (tree-level)', fontsize=13)
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    # Panel 2: Sim on its own scale (shows the shot-noise ridge)
    sim_max = max(abs(C_sim_2d.min()), abs(C_sim_2d.max()), 1e-15)
    im1 = axes[1].imshow(
        C_sim_2d.T, origin='lower', aspect='equal', extent=extent,
        cmap='RdBu_r', vmin=-sim_max, vmax=sim_max,
    )
    axes[1].set_xlabel(r'$\tau_1$', fontsize=13)
    axes[1].set_title('Sim (full scale)', fontsize=13)
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # Panel 3: Sim clamped to THEORY scale (reveals smooth structure)
    im2 = axes[2].imshow(
        C_sim_2d.T, origin='lower', aspect='equal', extent=extent,
        cmap='RdBu_r', vmin=-th_max, vmax=th_max,
    )
    axes[2].set_xlabel(r'$\tau_1$', fontsize=13)
    axes[2].set_title('Sim (theory scale)', fontsize=13)
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    fig.suptitle(rf'$k=3$ cumulant heatmap:  $\langle {corr_label} \rangle$',
                 fontsize=14)
    plt.tight_layout()
    plt.show()

else:
    print(f'Heatmap comparison only for k=3 (current k={k}).')
